# Mineração das unidades nº2 e nº10 — o contrato real das ferramentas

**Data:** 16–17/09/2026 · **Fonte:** `85cb11b5-b58b-40c4-a2cf-a3e99ac86521.csv.xz` (a mesma da análise genérica)
**Companheiro de:** [`analise_trace_esteira_juridica.ipynb`](analise_trace_esteira_juridica.ipynb) — esta análise
era a §11 daquele notebook até 18/09/2026, quando foi separada por ter virado um estudo próprio (pré-registrado,
com evidência por caso e produto final em `resultados/unidades_memoria.json`).

**Numeração preservada:** as seções continuam "§11.x", para manter a correspondência com as pastas
`resultados/evidencia/11.*` e com as referências já escritas nos documentos. Este notebook é **dono do
intervalo §11.x**: o próximo passe de análise (outro método, outras unidades) usa §12.x, e assim por diante —
número de seção nunca reutilizado, pasta de evidência nunca renomeada.

**Documentos:** lógica em [`../docs/06-racionais-mineracao-unidades-n2-n10.md`](../docs/06-racionais-mineracao-unidades-n2-n10.md)
§9 · números em [`../docs/07-relatorio-mineracao-unidades-n2-n10.md`](../docs/07-relatorio-mineracao-unidades-n2-n10.md) ·
conferência e desvios do pré-registro em
[`../docs/03-procedimento-validacao.md`](../docs/03-procedimento-validacao.md) §1.7–§1.11.






## 0 · Setup — a base compartilhada

O notebook não recopia código: [`base_pipeline.py`](base_pipeline.py), nesta pasta, é a espinha computacional
comum a qualquer análise do trace — carga (`carregar_trace`), explosão da working memory em steps
(`explodir_memoria` → `steps`, `RAW`, `execs`), classificação por causa-raiz (`classify` → `E`), mecanismos e
unidades (`submecanismo`, `UNI`, `EU`, `triagem`, `MIN_EXECS`, `MIN_MESES`) e a régua de sucesso verificado por
conteúdo (`medir_sucesso` → `S`, `DEGENERADO`). É o mesmo código das células de §§1–3 do notebook de análise
genérica — mudança lá muda os dois notebooks. Os outputs embutidos das §§11.x são da execução original de
16–17/09; re-rodar reproduz os mesmos números.


In [1]:
import json, re, ast, builtins, unicodedata
import pandas as pd, numpy as np
from collections import Counter, defaultdict
from base_pipeline import *

B = carregar_base()
df, steps, RAW, execs, E, EU, S = B.df, B.steps, B.RAW, B.execs, B.E, B.EU, B.S
print(f"base carregada: {len(df)} execuções · {len(steps):,} steps · {len(E)} erros · {len(UNI)} unidades")


base carregada: 1000 execuções · 5,781 steps · 498 erros · 14 unidades


## 11 · Mineração das unidades nº2 e nº10 — o contrato real das ferramentas

As unidades **nº2** ("Retorno das ferramentas de documento é dict") e **nº10** ("Campo inexistente no retorno
estruturado") são erros de leitura do retorno de uma ferramenta. Esta seção responde, em quatro passos, **qual
ferramenta produz cada erro, o que ela devolve de verdade, o que o system prompt diz que ela devolve, o que o agente
faz no step seguinte** (Passos 1–3), e depois se esse schema é estável, com que status, se bate com o trace cru numa
amostra, e qual o registro final de cada candidata (Passos 4–8).

Cada célula imprime só estrutura — nomes de função e de chave, tipos e contagens, nunca valores. As funções ficam
na primeira célula da seção (11.0). **Evidência:** cada análise grava `resultados/evidencia/<análise>/` com os casos
escolhidos por regra; `python drill_down.py evidencia <análise>` completa a pasta com os traces crus desses casos e uma
visão derivada de cada um, que só espelha trechos do cru (git-ignored, com PII). Regras de decisão (pré-registradas) e o porquê de cada uma:
[`../docs/06-racionais-mineracao-unidades-n2-n10.md`](../docs/06-racionais-mineracao-unidades-n2-n10.md) §9 · resultados comentados:
[`../docs/07-relatorio-mineracao-unidades-n2-n10.md`](../docs/07-relatorio-mineracao-unidades-n2-n10.md) §6.1 · conferência e os cinco logs de evidência:
[`../docs/03-procedimento-validacao.md`](../docs/03-procedimento-validacao.md) §1.7–1.8.




### 11.0 · Funções da seção

Só definições, sem saída: a maquinaria de AST e regex que os quatro passos usam. Não precisa ser lida para ler as
tabelas — o que cada regra faz e por quê está em [`../docs/06-racionais-mineracao-unidades-n2-n10.md`](../docs/06-racionais-mineracao-unidades-n2-n10.md) §9.




In [2]:
# Funções da §11 — só definições, sem saída. As células 11.1–11.4 chamam estas funções e mostram as tabelas.
# Regras de decisão (pré-registradas) e o porquê de cada uma: ../docs/06-racionais-mineracao-unidades-n2-n10.md §9.
# Entradas, produzidas no setup §0 deste notebook: RAW, EU / MIN_EXECS / MIN_MESES.
# Só estrutura sai daqui: nomes de função e de chave, tipos e contagens — nunca valores devolvidos pelas ferramentas
# nem nomes de variável (podem embutir identificadores). Chave com cara de dado vira <chave-dado>.
UNIDADES = {"U_contrato_dict": "nº2", "U_campo_inexistente": "nº10"}
EMBRULHO = {"list", "sorted", "reversed", "iter", "enumerate", "tuple"}
PEGA_TUDO = {"Exception", "BaseException", "KeyError", "LookupError", "IndexError", "TypeError"}
FALHOU = re.compile(r"^Code execution failed at line '(.*?)' due to: ", re.S)
MSG_INDEX = re.compile(r"Could not index (.*) with '(.*?)': (\w+(?:Error|Exception)): ", re.S)
SUGESTAO = re.compile(r"Maybe you meant one of these indexes instead: (\[.*?\])", re.S)
# 17/09: erro de 'AttributeError' sobre dict iterado como lista ('Object <chave> has no attribute get') não
# imprime o objeto inteiro, só a chave em que a iteração parou — regra geral, cruzada com o schema já
# derivado de OUTROS erros da mesma ferramenta (Passo 5), nunca específica de um caso (06-racionais-mineracao-unidades-n2-n10.md §9).
MSG_ATTR = re.compile(r"Object (.+?) has no attribute \w+", re.S)


def rotulo(u):
    return f"{UNIDADES.get(u, '?')} · {u}"


def _ordenar(t, erros="erros"):
    """nº2 antes de nº10; dentro da unidade, função com mais erros primeiro."""
    ordem = {rotulo(u): i for i, u in enumerate(UNIDADES)}
    nomes = list(t.index.names)
    t = t.reset_index()
    tot = t.groupby(["unidade", nomes[1]])[erros].transform("sum")
    t = t.assign(_u=t["unidade"].map(ordem), _f=-tot, _e=-t[erros]).sort_values(["_u", "_f", nomes[1], "_e"], kind="stable")
    return t.drop(columns=["_u", "_f", "_e"]).set_index(nomes)


def indexar_steps(raw):
    """(exec_id, role) → {idx: step}. O namespace Python persiste entre steps do mesmo papel."""
    passos = defaultdict(dict)
    for r in raw:
        passos[(r["exec_id"], r["role"])][r["idx"]] = r
    return passos


def erro_de(eu, a):
    return eu[(eu["exec_id"] == a["exec_id"]) & (eu["role"] == a["role"]) & (eu["idx"] == a["idx"])].iloc[0]


# ─────────────────────────────── Passo 1 — de qual ferramenta vem cada erro ───────────────────────────────

def pedido_da_msg(m):
    """O que o agente pediu, lido da mensagem: índice/chave, fatia, atributo ou colunas de DataFrame."""
    if "are in the [columns]" in m:
        return "colunas", None
    k = re.findall(r"KeyError: ('(?:[^'\\]|\\.)*'|\"(?:[^\"\\]|\\.)*\"|-?\d+)", m)
    if k:
        return "idx", ast.literal_eval(k[-1])
    if "unhashable type: 'slice'" in m:
        return "slice", None
    a = re.search(r"has no attribute (\w+)", m)
    return ("attr", a.group(1)) if a else (None, None)


def nos_que_pediram(stmt, tipo, valor):
    for n in ast.walk(stmt):
        if tipo == "idx" and isinstance(n, ast.Subscript) and isinstance(n.slice, ast.Constant) \
                and n.slice.value == valor:
            yield n.value
        elif tipo == "slice" and isinstance(n, ast.Subscript) and isinstance(n.slice, ast.Slice):
            yield n.value
        elif tipo == "colunas" and isinstance(n, ast.Subscript) and isinstance(n.slice, (ast.Name, ast.List)):
            yield n.value
        elif tipo == "attr" and isinstance(n, ast.Attribute) and n.attr == valor:
            yield n.value


def origem(e):
    while True:
        if isinstance(e, (ast.Subscript, ast.Attribute, ast.Starred)):
            e = e.value
        elif isinstance(e, ast.Call):
            f = e.func
            if isinstance(f, ast.Name) and f.id in EMBRULHO and e.args:
                e = e.args[0]
            elif isinstance(f, ast.Name):
                return "call", f.id
            elif isinstance(f, ast.Attribute) and isinstance(f.value, ast.Name) and e.args and (
                    f.value.id == "json" or (f.value.id in ("pd", "pandas") and f.attr in ("DataFrame", "json_normalize"))):
                e = e.args[0]
            elif isinstance(f, ast.Attribute):
                e = f.value
            else:
                return None, None
        elif isinstance(e, ast.Name):
            return "name", e.id
        else:
            return None, None


def alvos(t):
    return {x.id for x in ast.walk(t) if isinstance(x, ast.Name)}


def ligacoes(arvore, nome, antes_de=None):
    """Onde `nome` recebe valor: atribuição, for, comprehension. Devolve [(linha, expr_origem)]."""
    out = []
    for n in ast.walk(arvore):
        linha = getattr(n, "lineno", 0)
        if antes_de is not None and linha >= antes_de:
            continue
        if isinstance(n, ast.Assign) and any(nome in alvos(t) for t in n.targets):
            out.append((linha, n.value))
        elif isinstance(n, ast.AnnAssign) and n.value is not None and nome in alvos(n.target):
            out.append((linha, n.value))
        elif isinstance(n, (ast.For, ast.comprehension)) and nome in alvos(n.target):
            out.append((getattr(n, "lineno", getattr(n.iter, "lineno", 0)), n.iter))
    return out


def resolve(passos, idx, stmt_src, pedido):
    """Segue a variável que quebrou até a chamada que a criou: no próprio comando, antes dele no mesmo step,
    ou em steps anteriores do mesmo papel. Sem origem rastreável → 'não resolvido', nunca palpite."""
    try:
        stmt = ast.parse(stmt_src)
    except SyntaxError:
        return "não resolvido", "comando não parseia"
    cands = list(nos_que_pediram(stmt, *pedido))
    if not cands:
        return "não resolvido", "nó pedido não achado no comando"
    cod = passos[idx]["code"]
    try:
        arv = ast.parse(cod)
    except SyntaxError:
        arv = None
    linha_falha = None
    if arv is not None:
        primeira = stmt_src.strip().split("\n")[0].strip()
        for n in ast.walk(arv):
            if isinstance(n, ast.stmt):
                seg = ast.get_source_segment(cod, n) or ""
                if seg.strip().split("\n")[0].strip() == primeira:
                    linha_falha = n.lineno
                    break
    achados = set()
    caminho = set()
    for c in cands:
        tipo, nome = origem(c)
        vistos = set()
        # escopos em ordem: o próprio comando; o step antes da linha; steps anteriores inteiros
        escopo = ("comando", None)
        onde = "no próprio comando"
        while tipo == "name" and nome not in vistos and len(vistos) < 8:
            vistos.add(nome)
            achou = None
            if escopo[0] == "comando":
                lig = ligacoes(stmt, nome)
                if lig:
                    achou, onde = max(lig, key=lambda x: x[0])[1], "no próprio comando"
                    escopo = ("comando", None)
            if achou is None and arv is not None and linha_falha is not None:
                lim = linha_falha if escopo[0] == "comando" else escopo[1]
                if escopo[0] in ("comando", "step"):
                    lig = ligacoes(arv, nome, antes_de=lim)
                    if lig:
                        lin, achou = max(lig, key=lambda x: x[0])
                        onde, escopo = "no mesmo step", ("step", lin)
            if achou is None:
                for j in sorted((i for i in passos if i < idx), reverse=True):
                    try:
                        lig = ligacoes(ast.parse(passos[j]["code"]), nome)
                    except SyntaxError:
                        continue
                    if lig:
                        achou = max(lig, key=lambda x: x[0])[1]
                        onde, escopo = "em step anterior", ("anterior", None)
                        break
            if achou is None:
                break
            tipo, nome = origem(achou)
        if tipo == "call":
            achados.add(nome)
            caminho.add(onde)
    if len(achados) == 1:
        return achados.pop(), " + ".join(sorted(caminho))
    if len(achados) > 1:
        return "ambíguo: " + " / ".join(sorted(achados)), " + ".join(sorted(caminho))
    return "não resolvido", "variável sem origem rastreável"


def atribuir_origem(eu, passos, unidades):
    """Passo 1: uma linha por erro, com a função que produziu o objeto que quebrou."""
    linhas = []
    for _, e in eu[eu["unidade"].isin(unidades)].iterrows():
        m = FALHOU.search(e["err_msg"])
        pedido = pedido_da_msg(e["err_msg"])
        if not m or pedido[0] is None:
            fn, via = "não resolvido", "mensagem fora do formato esperado"
        else:
            fn, via = resolve(passos[(e["exec_id"], e["role"])], e["idx"], m.group(1), pedido)
        declaradas = set(re.findall(r"def\s+(\w+)\s*\(", passos[(e["exec_id"], e["role"])][e["idx"]]["sysprompt"] or ""))
        if fn in declaradas:
            natureza = "ferramenta declarada"
        elif fn.startswith(("não resolvido", "ambíguo")):
            natureza = fn.split(":")[0]
        else:
            natureza = "não declarada (auxiliar do agente / outra)"
        linhas.append({"exec_id": e["exec_id"], "role": e["role"], "idx": e["idx"], "mes": e["mes"],
                       "unidade": e["unidade"], "submecanismo": e["submecanismo"], "ocorrencia": e["ocorrencia"],
                       "funcao_origem": fn, "natureza": natureza, "via": via})
    return pd.DataFrame(linhas)


def veredito(atrib, unidade, nivel):
    """Régua pré-registrada: (a) dominante ≥90% → uma ferramenta só; (b) outra função com ≥10% → divide;
    (c) nem uma nem outra. `nivel` é 'erros' ou 'ocorrências' (cascata deduplicada)."""
    a = atrib[atrib["unidade"] == unidade]
    base = a if nivel == "erros" else a.drop_duplicates("ocorrencia")
    n = len(base)
    vc = base["funcao_origem"].value_counts()
    dom, dom_n = vc.index[0], vc.iloc[0]
    resolvidos = base[~base["funcao_origem"].str.startswith(("não resolvido", "ambíguo"))]
    outras = [f for f, c in resolvidos["funcao_origem"].value_counts().items() if f != dom and c / n >= 0.10]
    if dom_n / n >= 0.90:
        cod, texto = "a", "(a) uma ferramenta só (≥90%)"
    elif outras:
        cod, texto = "b", f"(b) divide — outra função com ≥10%: {', '.join(outras)}"
    else:
        cod, texto = "c", "(c) nem (a) nem (b) — sem estrutura clara"
    return cod, f"{rotulo(unidade)} [{nivel}]: {dom} = {dom_n}/{n} ({dom_n / n:.1%}) → {texto}"


def sub_unidades(atrib, min_execs, min_meses):
    """Cada (unidade, função) passa pela mesma triagem de recorrência do §9 e recebe um destino:
    candidata / documentar e monitorar (≥10% das ocorrências) / bucket de consulta (<10%)."""
    occ = atrib.drop_duplicates("ocorrencia")
    sub = occ.groupby(["unidade", "funcao_origem"]).agg(
        ocorrencias=("ocorrencia", "size"), execucoes=("exec_id", "nunique"), meses=("mes", "nunique"),
        papeis=("role", lambda s: ", ".join(sorted(set(s)))))
    sub.insert(0, "erros", atrib.groupby(["unidade", "funcao_origem"]).size())
    resolvida = ~sub.index.get_level_values("funcao_origem").str.startswith(("não resolvido", "ambíguo"))
    sub["passa triagem"] = np.where(resolvida, np.where((sub["execucoes"] >= min_execs) & (sub["meses"] >= min_meses), "sim", "não"), "—")
    # mesma regra para (a), (b) e (c); a fatia é medida em ocorrências, o mesmo nível que a triagem conta
    fatia = sub["ocorrencias"].values / occ.groupby("unidade").size().reindex(
        sub.index.get_level_values("unidade")).values
    sub["% ocorr. na unidade"] = (fatia * 100).round(1)
    sub["destino"] = [
        "documentar — atribuição a refazer" if passa == "—"
        else "candidata" if passa == "sim"
        else "documentar e monitorar (2ª extração)" if f >= 0.10
        else "bucket de consulta (drill_down)"
        for passa, f in zip(sub["passa triagem"], fatia)]
    return sub.sort_values(["unidade", "erros"], ascending=[True, False])


def tabela_sub_unidades(sub):
    return _ordenar(sub.rename(index=rotulo, level="unidade").rename_axis(["unidade", "função"]))


def bucket_consulta(atrib, sub):
    """Os erros das sub-unidades que não passam na triagem e cobrem <10% da unidade — para abrir caso a caso com
    `python drill_down.py caso <exec_id> <role>`."""
    no_bucket = set(sub.index[sub["destino"] == "bucket de consulta (drill_down)"])
    return atrib[[(u, fn) in no_bucket for u, fn in zip(atrib["unidade"], atrib["funcao_origem"])]][
        ["unidade", "exec_id", "role", "idx", "funcao_origem"]]


# ─────────────────────────────── Passo 2 — o que a ferramenta devolve de verdade ───────────────────────────────

def chave_segura(k):
    return k if isinstance(k, str) and re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]{0,40}", k) and not re.search(r"\d{5,}", k) else "<chave-dado>"


def chave_de_dado(k):
    return (chave_segura(k) == "<chave-dado>" or bool(re.fullmatch(r"[0-9a-f]{16,}", str(k)))
            or bool(re.fullmatch(r"[A-Z][A-Z0-9_]+", str(k))))


def forma(o, prof=0, mascarar=False):
    """Chaves e tipos, nunca valores. Listas mistas descrevem TODOS os tipos de elemento, não só o primeiro.
    Dicionário indexado por dado (hash, UF, categoria) ou com mais de 8 chaves vira resumo com as chaves
    mascaradas — nunca uma lista de valores."""
    if prof >= 5:
        return "…"
    if isinstance(o, dict):
        if not o:
            return "{}"
        if not mascarar and all(isinstance(v, dict) and v and all(isinstance(x, int) for x in v.values()) for v in o.values()):
            return "{<campo>: {<valor>: int}}"
        if mascarar or len(o) > 8 or sum(chave_de_dado(k) for k in o) / len(o) > 0.5:
            rot = "<vários campos>" if len(o) > 8 and not mascarar else "<chave>"
            return "{" + rot + ": " + " | ".join(sorted({forma(v, prof + 1, mascarar=True) for v in o.values()})) + "}"
        return "{" + ", ".join(f"{chave_segura(k)}: {forma(v, prof + 1)}" for k, v in sorted(o.items(), key=lambda kv: str(kv[0]))) + "}"
    if isinstance(o, (list, tuple)):
        if not o:
            return "[]"
        amostra = list(o[:50])
        registros = [x for x in amostra if isinstance(x, dict)]
        if not mascarar and registros and len(registros) == len(amostra) and all(
                len(d) <= 8 and not any(chave_de_dado(k) for k in d) for d in registros):
            campos = defaultdict(list)
            for d in registros:
                for k, v in d.items():
                    f = forma(v, prof + 1)
                    if f not in campos[k]:
                        campos[k].append(f)
            return "[{" + ", ".join(f"{chave_segura(k)}: {' | '.join(fs)}" for k, fs in sorted(campos.items())) + "}]"
        vistas = []
        for x in amostra:
            f = forma(x, prof + 1, mascarar)
            if f not in vistas:
                vistas.append(f)
        return "[" + " | ".join(vistas) + "]"
    return type(o).__name__


def ler_schema(atrib, eu, passos):
    """Passo 2: lê o objeto que a mensagem `Could not index {objeto} with '{chave}'` imprime, por dois métodos
    independentes (literal_eval × regex de nomes de chave), e reforça pelo `thought` (presença literal)."""
    linhas = []
    for _, a in atrib.iterrows():
        m = erro_de(eu, a)["err_msg"]
        step = passos[(a["exec_id"], a["role"])][a["idx"]]
        tipo_ped, pedido = pedido_da_msg(m)
        r = {"unidade": a["unidade"], "funcao_origem": a["funcao_origem"], "exec_id": a["exec_id"], "idx": a["idx"],
             "mes": a["mes"], "pedido": pedido if tipo_ped == "idx" else f"<{tipo_ped}>",
             "tem_valor": False, "literal_ok": False, "regex_ok": False, "chaves_topo": None, "forma": None,
             "chaves_regex": None, "concordam": None, "pedido_existe": None,
             "thought_cita_pedido": None, "thought_cita_funcao": a["funcao_origem"] in (step["thought"] or ""),
             "sugestao_smolagents": None, "chave_confirmada": None}
        mi = MSG_INDEX.search(m)
        if mi is None:
            ma = MSG_ATTR.search(m)
            if ma:
                r["chave_confirmada"] = chave_segura(ma.group(1).strip())
        if mi:
            valor = mi.group(1)
            r["tem_valor"] = True
            try:
                obj = ast.literal_eval(valor)
                r["literal_ok"] = True
                r["forma"] = forma(obj)
                if isinstance(obj, dict):
                    r["chaves_topo"] = tuple(sorted(chave_segura(k) for k in obj))
                    if tipo_ped == "idx":
                        r["pedido_existe"] = pedido in obj
            except Exception:
                pass
            achadas = re.findall(r"'(\w+)':", valor)
            if achadas:
                r["regex_ok"] = True
                r["chaves_regex"] = tuple(sorted({chave_segura(k) for k in achadas}))
            if r["literal_ok"] and r["regex_ok"] and r["chaves_topo"] is not None:
                r["concordam"] = set(r["chaves_topo"]) <= set(r["chaves_regex"]) and chave_segura(achadas[0]) == chave_segura(next(iter(obj)))
        if tipo_ped == "idx" and isinstance(pedido, str):
            r["thought_cita_pedido"] = pedido in (step["thought"] or "")
        sg = SUGESTAO.search(m)
        if sg:
            try:
                r["sugestao_smolagents"] = tuple(chave_segura(k) for k in ast.literal_eval(sg.group(1)))
            except Exception:
                r["sugestao_smolagents"] = ("<não parseou>",)
        linhas.append(r)
    return pd.DataFrame(linhas)


def chaves_reais(schema):
    """Os nomes de chave que aparecem na forma lida, por ferramenta (nunca chaves de dado)."""
    reais = defaultdict(set)
    for (_, fn), g in schema.groupby(["unidade", "funcao_origem"]):
        for f in g["forma"].dropna():
            reais[fn] |= set(re.findall(r"([A-Za-z_]\w*): ", f))
    return reais


def _conta(serie):
    s = serie.dropna()
    return f"{int(s.astype(bool).sum())}/{len(s)}" if len(s) else "—"


def tabela_leitura(schema):
    """Uma linha por (unidade, função): quantas mensagens trazem o objeto, quantas cada método leu, se concordam,
    e a sanidade (a chave pedida não pode existir no objeto — é o erro)."""
    linhas = []
    for (u, fn), g in schema.groupby(["unidade", "funcao_origem"], sort=False):
        cv = g[g["tem_valor"]]
        both = cv[cv["literal_ok"] & cv["regex_ok"]]
        sg = g["sugestao_smolagents"].dropna()
        linhas.append({
            "unidade": rotulo(u), "função": fn, "erros": len(g),
            "objeto na mensagem": f"{len(cv)}/{len(g)}",
            "literal_eval lê": f"{int(cv['literal_ok'].sum())}/{len(cv)}" if len(cv) else "—",
            "regex lê": f"{int(cv['regex_ok'].sum())}/{len(cv)}" if len(cv) else "—",
            "os dois concordam": _conta(both["concordam"]) if len(both) else "—",
            "chave pedida existe no objeto": _conta(cv["pedido_existe"]),
            "thought cita a chave pedida": _conta(g["thought_cita_pedido"]),
            "thought cita a função": f"{int(g['thought_cita_funcao'].sum())}/{len(g)}",
            "smolagents sugere chave": f"{len(sg)}/{len(g)}"})
    return _ordenar(pd.DataFrame(linhas).set_index(["unidade", "função"]))


def tabela_forma(schema):
    """A forma (chaves e tipos) do objeto indexado e o que o agente pediu a ele."""
    t = schema.assign(unidade=schema["unidade"].map(rotulo),
                      forma=schema["forma"].fillna("(objeto não lido da mensagem)"),
                      pedido=schema["pedido"].astype(str))
    return _ordenar(t.groupby(["unidade", "funcao_origem", "forma", "pedido"]).size().to_frame("erros")
                     .rename_axis(["unidade", "função", "forma do objeto indexado", "pedido do agente"]))


# ─────────────────────── Addendum ao Passo 2 — o que o system prompt diz que a ferramenta devolve ───────────────────────

def presente(texto, chave):
    """Presença literal em três formas, da mais frouxa à mais estrita: palavra solta, entre aspas, chave de JSON."""
    k = re.escape(chave)
    return (bool(re.search(r"\b" + k + r"\b", texto)),
            bool(re.search(r"['\"]" + k + r"['\"]", texto)),
            bool(re.search(r"['\"]" + k + r"['\"]\s*:", texto)))


def bloco_da_ferramenta(prompt, fn):
    m = re.search(r"def\s+" + re.escape(fn) + r"\s*\(", prompt)
    if not m:
        return ""
    fim = re.search(r"\ndef\s+\w+\s*\(", prompt[m.end():])
    return prompt[m.start(): m.end() + fim.start()] if fim else prompt[m.start():]


def retorno_declarado(bloco):
    """(assinatura, o que o bloco diz sobre o retorno): a seção `Outputs:` quando existe; senão, as frases da
    descrição que falam em 'return'. Texto literal da documentação da ferramenta, não dado de caso."""
    assinatura = bloco.split("\n")[0].strip()
    m = re.search(r"Outputs?:\s*\n(.*?)(?:\n\s*\n|$)", bloco, re.S)
    if m:
        return assinatura, " ".join(m.group(1).split())
    doc = re.search(r'"""(.*?)(?:\n\s*Args:|""")', bloco, re.S)
    frases = re.split(r"(?<=[.!?])\s+", " ".join(doc.group(1).split())) if doc else []
    return assinatura, " ".join(f for f in frases if re.search(r"\breturns?\b|\bretorna", f, re.I)) or "(o bloco não descreve o retorno)"


def tabela_retorno_declarado(atrib, passos, reais):
    """Uma linha por ferramenta com chave real minerada: assinatura e retorno declarados no prompt, quantas
    variantes do bloco existem nos steps com erro, e o retorno real (Passo 2) para comparar."""
    linhas = []
    for fn, g in atrib.groupby("funcao_origem", sort=False):
        if not reais.get(fn):
            continue
        blocos = Counter(bloco_da_ferramenta(passos[(a["exec_id"], a["role"])][a["idx"]]["sysprompt"] or "", fn)
                         for _, a in g.iterrows())
        bloco, _ = blocos.most_common(1)[0]
        assinatura, retorno = retorno_declarado(bloco) if bloco else ("(não declarada)", "")
        linhas.append({"função": fn, "erros (nas duas unidades)": len(g), "variantes do bloco nesses erros": len(blocos),
                       "assinatura no prompt": assinatura, "retorno declarado no prompt": retorno})
    return pd.DataFrame(linhas).set_index("função")


def presenca_no_prompt(atrib, eu, passos, reais):
    """Uma linha por chave (as reais do Passo 2 e a errada que o agente pediu): em quantos system prompts dos steps
    com erro ela aparece — solta, entre aspas, como chave de JSON — e se está dentro do bloco da própria ferramenta."""
    linhas = []
    for (u, fn), g in atrib.groupby(["unidade", "funcao_origem"], sort=False):
        chaves = sorted(reais.get(fn, set()))
        if not chaves:
            continue
        prompts = [passos[(a["exec_id"], a["role"])][a["idx"]]["sysprompt"] or "" for _, a in g.iterrows()]
        blocos = [bloco_da_ferramenta(p, fn) for p in prompts]
        declaradas = Counter(k for b in blocos for k in set(re.findall(r"['\"](\w+)['\"]\s*:", b)))
        erradas = sorted({p[1] for p in (pedido_da_msg(erro_de(eu, a)["err_msg"]) for _, a in g.iterrows())
                          if p[0] == "idx" and isinstance(p[1], str)})
        for tipo, lista in (("real", chaves), ("errada (pedida pelo agente)", erradas)):
            for c in lista:
                pres = [presente(p, c) for p in prompts]
                linhas.append({"unidade": rotulo(u), "função": fn, "chave": c, "tipo": tipo,
                               "palavra solta": sum(x[0] for x in pres), "entre aspas": sum(x[1] for x in pres),
                               "como chave (':')": sum(x[2] for x in pres),
                               "no bloco da ferramenta": declaradas.get(c, 0), "de": len(prompts)})
    return _ordenar(pd.DataFrame(linhas).assign(_n=lambda d: d["de"]).set_index(["unidade", "função", "chave"]),
                    erros="_n").drop(columns="_n")


# ─────────────────────────────── Passo 3 — o que o agente fez no step seguinte ───────────────────────────────

def variaveis_indexadas(stmt_src, pedido):
    try:
        stmt = ast.parse(stmt_src)
    except SyntaxError:
        return set()
    return {nome for t, nome in (origem(c) for c in nos_que_pediram(stmt, *pedido)) if t == "name"}


def protegidos_por_try(arv):
    ids = set()
    for n in ast.walk(arv):
        if isinstance(n, ast.Try):
            pega = any(h.type is None or any(isinstance(x, ast.Name) and x.id in PEGA_TUDO
                                             for x in ast.walk(h.type)) for h in n.handlers)
            if pega:
                ids |= {id(x) for b in n.body for x in ast.walk(b)}
    return ids


def segundo_parametro(call):
    """O segundo parâmetro de .get(chave, padrão), só como estrutura: literal curto; leitura de outra chave
    (<var>.get('k') / <var>['k']); ou o tipo do nó. Nome de variável nunca sai (pode embutir identificador)."""
    if len(call.args) < 2:
        return "sem padrão"
    d = call.args[1]
    if isinstance(d, ast.Constant) and len(repr(d.value)) <= 20 and not re.search(r"\d{5,}", repr(d.value)):
        return repr(d.value)
    if (isinstance(d, ast.Call) and isinstance(d.func, ast.Attribute) and d.func.attr == "get" and d.args
            and isinstance(d.args[0], ast.Constant)):
        return f"<var>.get('{chave_segura(d.args[0].value)}')"
    if isinstance(d, ast.Subscript) and isinstance(d.slice, ast.Constant):
        return f"<var>['{chave_segura(d.slice.value)}']"
    return f"<{type(d).__name__}>"


def em_ramo_guardado(arv, var):
    """ids dos nós dentro de if/else (ou expressão x if c else y) cujo teste olha a própria `var` —
    ex.: docs['result'][0] if 'result' in docs else docs[0]."""
    ids = set()
    for n in ast.walk(arv):
        if isinstance(n, (ast.If, ast.IfExp)) and any(isinstance(x, ast.Name) and x.id == var for x in ast.walk(n.test)):
            ramos = (n.body + n.orelse) if isinstance(n, ast.If) else [n.body, n.orelse]
            ids |= {id(x) for r in ramos for x in ast.walk(r)}
    return ids


def acessos(arv, var, fn):
    """[(forma, chave, protegido, padrão, guardado)] para as leituras de `var` que ainda se referem a um objeto
    com o schema que quebrou: se o step reatribui `var` a outra coisa (ex.: docs = docs['result'][0]), só contam as
    leituras antes disso e as do lado direito da reatribuição. Reatribuir chamando a mesma ferramenta não corta."""
    prot = protegidos_por_try(arv)
    guard = em_ramo_guardado(arv, var)
    ligs = [n for n in ast.walk(arv)
            if (isinstance(n, (ast.Assign, ast.AnnAssign, ast.AugAssign)) and any(
                isinstance(x, ast.Name) and x.id == var and isinstance(x.ctx, ast.Store)
                for t in (n.targets if isinstance(n, ast.Assign) else [n.target]) for x in ast.walk(t))
                and not (n.value is not None and origem(n.value) == ("call", fn)))
            or (isinstance(n, ast.For) and any(isinstance(x, ast.Name) and x.id == var for x in ast.walk(n.target)))]
    corte, dentro = None, set()
    if ligs:
        b = min(ligs, key=lambda n: (n.lineno, n.col_offset))
        corte = b.lineno
        valor = b.iter if isinstance(b, ast.For) else b.value
        dentro = {id(x) for x in ast.walk(valor)} if valor is not None else set()
    out = []
    for n in ast.walk(arv):
        if corte is not None and id(n) not in dentro and getattr(n, "lineno", 0) >= corte:
            continue
        if isinstance(n, ast.Subscript) and not isinstance(n.ctx, ast.Load):
            continue
        if isinstance(n, ast.Subscript) and isinstance(n.value, ast.Name) and n.value.id == var:
            chave = n.slice.value if isinstance(n.slice, ast.Constant) else ("<fatia>" if isinstance(n.slice, ast.Slice) else "<expr>")
            out.append(("[]", chave, id(n) in prot, None, id(n) in guard))
        elif (isinstance(n, ast.Call) and isinstance(n.func, ast.Attribute) and n.func.attr == "get"
              and isinstance(n.func.value, ast.Name) and n.func.value.id == var and n.args):
            chave = n.args[0].value if isinstance(n.args[0], ast.Constant) else "<expr>"
            out.append((".get", chave, id(n) in prot, segundo_parametro(n), id(n) in guard))
    return out


def _chaves_novas(outras):
    novas = sorted({str(k) for _, k, _, _, _ in outras})
    return novas, ", ".join(chave_segura(k) if not k.startswith("<") and not k.lstrip("-").isdigit() else k for k in novas)


def classificar_conserto(atrib, eu, passos, reais):
    """Passo 3: (1) troca de chave; (2) conserto silencioso — a mesma chave errada protegida por .get(chave, padrão)
    ou try/except; (3) sem conserto comparável. Só estrutura: chaves e o segundo parâmetro do .get."""
    linhas = []
    for _, a in atrib.iterrows():
        e = erro_de(eu, a)
        ps = passos[(a["exec_id"], a["role"])]
        tipo_ped, pedido = pedido_da_msg(e["err_msg"])
        ms = FALHOU.search(e["err_msg"])
        vars_ = variaveis_indexadas(ms.group(1), (tipo_ped, pedido)) if ms and tipo_ped else set()
        r = {"unidade": a["unidade"], "funcao_origem": a["funcao_origem"], "exec_id": a["exec_id"], "role": a["role"],
             "idx": a["idx"], "pedido": pedido if tipo_ped == "idx" else f"<{tipo_ped}>",
             "variavel": ", ".join(sorted(re.sub(r"\d{5,}", "#", v) for v in vars_)),
             "tipo": None, "detalhe": None, "chaves_novas": None, "padrao": None, "bate_schema": None, "seguinte_com_erro": None}
        prox = ps.get(a["idx"] + 1)
        if tipo_ped in ("attr", "colunas", None):
            r["tipo"], r["detalhe"] = "não aplicável", f"erro de {tipo_ped or 'formato desconhecido'}, não de chave/índice"
        elif prox is None:
            r["tipo"], r["detalhe"] = "3 · sem conserto comparável", "não há step seguinte"
        elif not vars_:
            r["tipo"], r["detalhe"] = "3 · sem conserto comparável", "variável indexada não identificada"
        else:
            r["seguinte_com_erro"] = bool(prox["err_type"])
            try:
                arv = ast.parse(prox["code"])
            except SyntaxError:
                arv = None
            if arv is None:
                r["tipo"], r["detalhe"] = "3 · sem conserto comparável", "código do step seguinte não parseia"
            else:
                errado = "<fatia>" if tipo_ped == "slice" else pedido
                acc = [x for v in vars_ for x in acessos(arv, v, a["funcao_origem"])]
                mesma = [x for x in acc if x[1] == errado]
                outras = [x for x in acc if x[1] != errado and x[1] != "<expr>"]
                chaves_fn = reais.get(a["funcao_origem"], set())
                if any(f == ".get" or prot for f, _, prot, _, _ in mesma):
                    r["tipo"] = "2 · conserto silencioso"
                    r["detalhe"] = " + ".join(sorted({".get" if f == ".get" else "try/except" for f, _, prot, _, _ in mesma if f == ".get" or prot}))
                    r["padrao"] = ", ".join(sorted({p for f, _, _, p, _ in mesma if f == ".get"})) or None
                elif mesma and outras and all(g for *_, g in mesma):
                    novas, r["chaves_novas"] = _chaves_novas(outras)
                    r["tipo"] = "1 · troca de chave"
                    r["bate_schema"] = any(k in chaves_fn for k in novas) if chaves_fn else None
                    r["detalhe"] = "com guarda de tipo — o acesso antigo fica num ramo alternativo"
                elif mesma:
                    r["tipo"], r["detalhe"] = "fora dos três tipos · repetiu o acesso errado", "mesma chave, sem proteção nem guarda"
                elif outras:
                    novas, r["chaves_novas"] = _chaves_novas(outras)
                    r["tipo"] = "1 · troca de chave"
                    r["bate_schema"] = any(k in chaves_fn for k in novas) if chaves_fn else None
                    r["detalhe"] = "confirma o schema da ferramenta (Passo 2)" if r["bate_schema"] else ("não bate com o schema" if chaves_fn else "sem schema minerado")
                else:
                    r["tipo"], r["detalhe"] = "3 · sem conserto comparável", "step seguinte não lê mais a variável"
        linhas.append(r)
    return pd.DataFrame(linhas)


def tabela_desfechos(autocorr):
    """Uma linha por (unidade, função): quantos erros caíram em cada desfecho do Passo 3."""
    t = autocorr.copy()
    t["desfecho"] = np.where(t["detalhe"].eq("com guarda de tipo — o acesso antigo fica num ramo alternativo"),
                             "1 · troca de chave (com guarda de tipo)",
                             np.where(t["tipo"].eq("1 · troca de chave"), "1 · troca de chave (sem guarda)", t["tipo"]))
    ordem = ["1 · troca de chave (sem guarda)", "1 · troca de chave (com guarda de tipo)", "2 · conserto silencioso",
             "3 · sem conserto comparável", "fora dos três tipos · repetiu o acesso errado", "não aplicável"]
    piv = (t.assign(unidade=t["unidade"].map(rotulo))
            .pivot_table(index=["unidade", "funcao_origem"], columns="desfecho", values="idx", aggfunc="size", fill_value=0)
            .reindex(columns=ordem, fill_value=0))
    trocas = t[t["tipo"].eq("1 · troca de chave")].assign(unidade=lambda d: d["unidade"].map(rotulo))
    g = trocas.groupby(["unidade", "funcao_origem"])
    piv.insert(0, "erros", t.assign(unidade=t["unidade"].map(rotulo)).groupby(["unidade", "funcao_origem"]).size())
    piv["trocas que batem com o schema"] = g["bate_schema"].apply(lambda s: int(s.fillna(False).astype(bool).sum()))
    piv["trocas com step seguinte sem erro"] = g["seguinte_com_erro"].apply(lambda s: int((~s.astype(bool)).sum()))
    piv[["trocas que batem com o schema", "trocas com step seguinte sem erro"]] = \
        piv[["trocas que batem com o schema", "trocas com step seguinte sem erro"]].fillna(0).astype(int)
    return _ordenar(piv.rename_axis(index=["unidade", "função"], columns=None))


def tabela_detalhe(autocorr):
    """O detalhe de cada desfecho: a chave nova lida (troca), o segundo parâmetro do .get (conserto silencioso), ou o
    motivo (sem conserto / não aplicável)."""
    t = autocorr.assign(unidade=autocorr["unidade"].map(rotulo),
                        chave_ou_padrao=autocorr["chaves_novas"].fillna(autocorr["padrao"]).fillna("—"))
    return _ordenar(t.groupby(["unidade", "funcao_origem", "tipo", "detalhe", "chave_ou_padrao"]).size().to_frame("erros")
                     .rename_axis(["unidade", "função", "desfecho", "detalhe", "chave nova / 2º parâmetro do .get"]))

# ─────────────────────────────── Evidência por análise ───────────────────────────────
# resultados/evidencia/<análise>/ — git-ignored. O notebook escreve só estrutura: casos.csv (os casos escolhidos por
# regra, com o motivo), derivados/<tabela>.csv (uma linha por erro, com exec_id/role/idx para voltar ao cru) e
# leia-me.md. Os traces crus (crus/<exec_id>.json, a linha inteira do trace) e a visão de cada caso
# (derivados/<exec>_<role>_idx<n>.json, trechos do cru com o caminho de cada um) são escritos por
# `python drill_down.py evidencia <análise>`, que confere cada trecho contra o cru ao gravar.
import os
import random

EVIDENCIA = "resultados/evidencia"
ORDEM = ["mes", "exec_id", "idx"]


def primeiro_por(t, chaves):
    """O primeiro caso de cada grupo na ordem (mês, exec_id, idx) — regra fixa, não escolha a dedo."""
    return t.sort_values(ORDEM).drop_duplicates(chaves)


def registrar_evidencia(analise, secao, afirmacao, regra, casos, tabelas, nota=""):
    pasta = os.path.join(EVIDENCIA, analise)
    os.makedirs(os.path.join(pasta, "derivados"), exist_ok=True)
    outras = {k: "first" for k in casos.columns if k not in ("exec_id", "role", "idx", "motivo")}
    c = casos.groupby(["exec_id", "role", "idx"], sort=False, as_index=False).agg(
        {**outras, "motivo": lambda s: "; ".join(dict.fromkeys(s))})
    c.insert(0, "analise", analise)
    c.to_csv(os.path.join(pasta, "casos.csv"), index=False)
    for nome, t in tabelas.items():
        t.to_csv(os.path.join(pasta, "derivados", f"{nome}.csv"), index=False)
    arquivos = "".join(f"- `derivados/{n}.csv` — tabela da análise, com `exec_id`/`role`/`idx` por linha para voltar ao "
                       "cru.\n" for n in tabelas)
    with open(os.path.join(pasta, "leia-me.md"), "w", encoding="utf-8") as f:
        f.write(
            f"# {analise} — notebook {secao}\n\n"
            f"**O que esta pasta sustenta.** {afirmacao}\n\n"
            f"**Regra de escolha dos {len(c)} casos.** {regra}\n\n"
            + (f"{nota}\n\n" if nota else "")
            + "**Arquivos.**\n\n"
            "- `crus/<exec_id>.json` — **a fonte**: a linha inteira do trace, sem alteração (`txt_etap_memo` só "
            "desserializado).\n"
            "- `derivados/<exec>_<role>_idx<n>.json` — visão de um caso para leitura humana: cada trecho copiado traz o "
            "caminho no cru, e os campos calculados dizem a regra. Em dúvida, vale o cru.\n"
            "- `casos.csv` — os casos escolhidos, com o motivo e as colunas calculadas pelo notebook.\n"
            + arquivos
            + f"\n**Gerar os crus e as visões por caso** (em `pipeline/`): `uv run python drill_down.py evidencia {analise}`\n\n"
            "> Os crus têm nome de cliente, número de processo e texto de documento. Pasta git-ignored — não versionar.\n")
    print(f"evidência: {len(c)} casos e {len(tabelas)} tabela(s) em {pasta}/ — crus e visões por caso: "
          f"`python drill_down.py evidencia {analise}`")


def encurtar_ids(o):
    if isinstance(o, dict):
        return {k: (v[:8] + "…" if k == "exec_id" else encurtar_ids(v)) for k, v in o.items()}
    if isinstance(o, list):
        return [encurtar_ids(x) for x in o]
    return o


# ─────────────────────────────── Passo 4 — o schema é o mesmo em todos os meses? ───────────────────────────────

def chaves_da_forma(f):
    return set(re.findall(r"([A-Za-z_]\w*): ", f))


def comparar_forma(f, comum):
    """idêntica; campos a mais (a forma comum inteira está lá, mais chaves de tipo simples); ou contradição (falta chave
    da forma comum, ou o tipo/aninhamento muda)."""
    if f == comum:
        return "idêntica"
    extras = chaves_da_forma(f) - chaves_da_forma(comum)
    if not extras or not chaves_da_forma(comum) <= chaves_da_forma(f):
        return "contradição"
    sem = f
    for k in extras:
        sem = re.sub(r"\b" + re.escape(k) + r": [A-Za-z_]+(?: \| [A-Za-z_]+)*(, )?", "", sem)
    sem = re.sub(r", ([}\]])", r"\1", sem)
    return "campos a mais" if sem == comum else "contradição"


def estabilidade(sch, candidatas):
    """Por sub-unidade candidata e nível (as chaves de topo do objeto lido), cada forma comparada com a mais comum."""
    sel = pd.Series([(u, fn) in candidatas for u, fn in zip(sch["unidade"], sch["funcao_origem"])], index=sch.index)
    t = sch[sel & sch["literal_ok"].astype(bool)].copy()
    t["nivel"] = t["chaves_topo"].map(lambda k: "{" + ", ".join(k) + "}")
    comum = t.groupby(["unidade", "funcao_origem", "nivel"])["forma"].agg(lambda s: s.value_counts().index[0])
    t["forma_comum"] = [comum[k] for k in zip(t["unidade"], t["funcao_origem"], t["nivel"])]
    t["comparacao"] = [comparar_forma(f, c) for f, c in zip(t["forma"], t["forma_comum"])]
    return t


def chaves_do_schema(est, u, fn):
    """As chaves da forma comum do nível mais frequente da sub-unidade."""
    e = est[(est["unidade"] == u) & (est["funcao_origem"] == fn)]
    nivel = e["nivel"].value_counts().index[0]
    return nivel, e.loc[e["nivel"] == nivel, "forma_comum"].iloc[0]


def _meses(s):
    return ", ".join(sorted(set(s))) or "—"


def tabela_meses(est):
    t = est.assign(unidade=est["unidade"].map(rotulo))
    p = t.pivot_table(index=["unidade", "funcao_origem", "nivel", "comparacao"], columns="mes", values="idx",
                      aggfunc="size", fill_value=0)
    p = p.rename_axis(index=["unidade", "função", "nível (chaves de topo)", "comparação com a forma comum"], columns=None)
    p.insert(0, "erros", p.sum(axis=1))
    return _ordenar(p)


def tabela_estabilidade(est):
    linhas = []
    for (u, fn, nv), g in est.groupby(["unidade", "funcao_origem", "nivel"]):
        var, con = g[g["comparacao"] == "campos a mais"], g[g["comparacao"] == "contradição"]
        meses_comum = set(g.loc[g["comparacao"] == "idêntica", "mes"])
        if len(con):
            v = f"schema mudou em {_meses(con['mes'])}"
        elif len(var):
            junto = " (a forma comum também aparece nesse(s) mês(es))" if set(var["mes"]) <= meses_comum else ""
            v = f"estável, com campos a mais em {_meses(var['mes'])}{junto}"
        else:
            v = "estável"
        difere = g.loc[g["forma"] != g["forma_comum"], "mes"]
        linhas.append({"unidade": rotulo(u), "função": fn, "nível (chaves de topo)": nv, "erros": len(g),
                       "meses observados": _meses(g["mes"]), "nº de meses": g["mes"].nunique(),
                       "formas distintas": g["forma"].nunique(), "idênticas": int((g["comparacao"] == "idêntica").sum()),
                       "campos a mais": len(var), "contradições": len(con), "veredito": v,
                       "leitura estrita (qualquer diferença conta)": "estável" if difere.empty else f"difere em {_meses(difere)}"})
    return _ordenar(pd.DataFrame(linhas).set_index(["unidade", "função", "nível (chaves de topo)"]))


# ─────────────────────────────── Passo 5 — status ───────────────────────────────

def status_passo5(sch, est, candidatas, chaves_reais):
    """derived-and-checked se ≥90% das ocorrências tiveram o objeto lido no Passo 2 OU confirmado por evidência
    indireta (17/09 — mensagem de AttributeError cuja chave bate com o schema já derivado de OUTROS erros da mesma
    ferramenta; regra geral, não específica de nenhum caso) e o Passo 4 não achou contradição; senão parcial. Uma
    ocorrência conta como lida/confirmada se pelo menos um dos seus erros contar. A leitura estrita (só objeto lido de
    fato, sem a confirmação indireta) fica reportada ao lado."""
    linhas, residuo = [], []
    for u, fn in candidatas:
        g = sch[(sch["unidade"] == u) & (sch["funcao_origem"] == fn)].copy()
        reais = chaves_reais.get(fn, set())
        g["confirmado"] = g["literal_ok"].astype(bool) | (g["chave_confirmada"].notna() & g["chave_confirmada"].isin(reais))
        occ = g.groupby("ocorrencia")["confirmado"].any()
        occ_estrita = g.groupby("ocorrencia")["literal_ok"].any()
        e = est[(est["unidade"] == u) & (est["funcao_origem"] == fn)]
        n_con, n_var = int((e["comparacao"] == "contradição").sum()), int((e["comparacao"] == "campos a mais").sum())
        cob, cob_estrita = occ.mean(), occ_estrita.mean()
        linhas.append({"unidade": rotulo(u), "função": fn, "erros": len(g),
                       "erros lidos ou confirmados": int(g["confirmado"].sum()),
                       "confirmados por AttributeError": int((g["confirmado"] & ~g["literal_ok"].astype(bool)).sum()),
                       "ocorrências": len(occ), "ocorrências lidas/confirmadas": int(occ.sum()), "cobertura": f"{cob:.1%}",
                       "cobertura estrita": f"{cob_estrita:.1%}",
                       "contradições (Passo 4)": n_con, "campos a mais (Passo 4)": n_var,
                       "status": "derived-and-checked" if cob >= 0.90 and n_con == 0 else "parcial",
                       "status na leitura estrita": "derived-and-checked" if cob_estrita >= 0.90 and n_con + n_var == 0 else "parcial"})
        r = g[~g["confirmado"]]
        motivo = pd.Series(["a mensagem não imprime o objeto nem confirma chave conhecida (pedido " + str(p) + ")" if not tv
                            else "o objeto impresso não foi lido" for p, tv in zip(r["pedido"], r["tem_valor"])],
                           dtype=object, index=r.index)
        residuo.append(r.assign(motivo_residuo=motivo))
    return _ordenar(pd.DataFrame(linhas).set_index(["unidade", "função"])), pd.concat(residuo)


# ─────────────────────────────── Passo 6 — amostra para verificação contra o trace cru ───────────────────────────────
SEMENTE_P6 = 20260917


def amostra_passo6(sch, candidatas, n=5, semente=SEMENTE_P6):
    """n erros por sub-unidade, sorteados com semente fixa: 1 do resíduo do Passo 5, se houver, e o resto entre os lidos."""
    rng = random.Random(semente)
    escolhidos = []
    for u, fn in candidatas:
        g = sch[(sch["unidade"] == u) & (sch["funcao_origem"] == fn)].sort_values(ORDEM)
        lidos = list(g.index[g["literal_ok"].astype(bool)])
        residuo = list(g.index[~g["literal_ok"].astype(bool)])
        if residuo:
            escolhidos.append((residuo[rng.randrange(len(residuo))], "resíduo do Passo 5"))
        k = min(n - (1 if residuo else 0), len(lidos))
        escolhidos += [(i, "sorteado entre os erros com objeto lido") for i in rng.sample(lidos, k)]
    return pd.concat([sch.loc[[i]].assign(motivo=m) for i, m in escolhidos])


def logs_de_print(passos, exec_id, role, idx):
    """Os logs de print (Execution logs) que o agente recebeu até o step seguinte ao erro — sem as mensagens de erro,
    para ser uma fonte diferente do objeto que a mensagem imprime."""
    ps = passos[(exec_id, role)]
    st = ps.get(idx + 1) or ps[idx]
    try:
        msgs = json.loads(st["ctx"]) if st["ctx"] else []
    except Exception:
        return ""
    logs = []
    for m in msgs:
        if not isinstance(m, dict) or m.get("role") != "tool-response":
            continue
        c = m.get("content")
        s = "".join(x.get("text", "") for x in c if isinstance(x, dict)) if isinstance(c, list) else str(c or "")
        if "Execution logs:" in s and "Code execution failed" not in s and "Could not index" not in s:
            logs.append(s)
    return "\n".join(logs)


def conferir_amostra(amostra, est, autocorr, passos):
    """Três fontes por caso: (1) o objeto da mensagem tem a forma comum (mesmo leitor do Passo 2 — consistência, não
    acerto); (2) as chaves do schema aparecem no log de print; (3) o conserto do Passo 3 lê uma chave do schema."""
    linhas = []
    ordem_u = amostra["unidade"].map(list(UNIDADES).index).rename("_u")
    for i, a in amostra.join(ordem_u).sort_values(["_u", *ORDEM]).iterrows():
        nivel, comum = chaves_do_schema(est, a["unidade"], a["funcao_origem"])
        chaves = sorted(chaves_da_forma(comum))
        if not a["literal_ok"]:
            c1 = "— (objeto não lido: resíduo)"
        elif "{" + ", ".join(a["chaves_topo"]) + "}" != nivel:
            c1 = "outro nível: {" + ", ".join(a["chaves_topo"]) + "}"
        else:
            c1 = {"idêntica": "sim", "campos a mais": "sim, com campos a mais", "contradição": "não"}[comparar_forma(a["forma"], comum)]
        logs = logs_de_print(passos, a["exec_id"], a["role"], a["idx"])
        no_log = [k for k in chaves if re.search(r"['\"]" + re.escape(k) + r"['\"]\s*:", logs)]
        c2 = "sem log de print" if not logs else (f"sim ({len(no_log)}/{len(chaves)})" if len(no_log) == len(chaves)
                                                  else f"parcial ({len(no_log)}/{len(chaves)})")
        ac = autocorr.loc[i]
        if ac["tipo"] == "1 · troca de chave":
            c3 = ("sim — lê " if ac["bate_schema"] else "não — lê ") + str(ac["chaves_novas"])
        elif ac["tipo"] == "2 · conserto silencioso":
            c3 = f".get com padrão {ac['padrao']}"
        else:
            c3 = f"— ({ac['tipo']})"
        linhas.append({"exec_id": a["exec_id"], "role": a["role"], "unidade": rotulo(a["unidade"]), "função": a["funcao_origem"],
                       "mês": a["mes"], "exec": a["exec_id"][:8] + "…", "idx": a["idx"], "motivo": a["motivo"],
                       "(1) objeto da mensagem tem a forma comum": c1,
                       "(2) chaves do schema no log de print": c2,
                       "(3) conserto lê chave do schema": c3})
    return pd.DataFrame(linhas)


# ─────────────────────────────── Passos 7 e 8 — o registro final ───────────────────────────────

def _alternando_meses(t, n):
    grupos = [g.sort_values(ORDEM) for _, g in t.groupby("mes", sort=True)]
    out, i = [], 0
    while len(out) < n and any(i < len(g) for g in grupos):
        out += [g.iloc[i] for g in grupos if i < len(g)][: n - len(out)]
        i += 1
    return out


def acesso_do_conserto(u, fn, atrib, eu, autocorr, passos):
    """O acesso completo (cadeia de índices constantes) que o próprio agente escreveu na variável que quebrou, nos
    consertos de troca de chave cujo step seguinte roda sem erro — ex.: ['result'][0]. Conta um por conserto."""
    cont, n = Counter(), 0
    for i, a in atrib[(atrib["unidade"] == u) & (atrib["funcao_origem"] == fn)].iterrows():
        ac = autocorr.loc[i]
        if ac["tipo"] != "1 · troca de chave" or ac["seguinte_com_erro"]:
            continue
        e = erro_de(eu, a)
        tipo_ped, pedido = pedido_da_msg(e["err_msg"])
        ms = FALHOU.search(e["err_msg"])
        vars_ = variaveis_indexadas(ms.group(1), (tipo_ped, pedido)) if ms else set()
        try:
            arv = ast.parse(passos[(a["exec_id"], a["role"])][a["idx"] + 1]["code"])
        except SyntaxError:
            continue
        subs = [x for x in ast.walk(arv) if isinstance(x, ast.Subscript) and isinstance(x.ctx, ast.Load)]
        internos = {id(x.value) for x in subs}
        cadeias = set()
        for x in subs:
            if id(x) in internos:
                continue
            cadeia, y = [], x
            while isinstance(y, ast.Subscript) and isinstance(y.slice, ast.Constant):
                cadeia.append(y.slice.value)
                y = y.value
            if isinstance(y, ast.Name) and y.id in vars_ and cadeia:
                cadeia.reverse()
                cadeias.add("".join(f"['{chave_segura(k)}']" if isinstance(k, str) else f"[{k}]" for k in cadeia))
        n += 1
        cont.update(cadeias)
    return cont, n


def registro_final(u, fn, atrib, eu, sch, autocorr, presenca, est, status, conf, passos, desfecho=None):
    """O registro no schema de 01-racionais.md §8, só com o que os Passos 1–6 derivaram. description e
    correction_guidance saem de um molde fixo (Passo 7): o mesmo fato em modo indicativo e em modo imperativo."""
    a = atrib[(atrib["unidade"] == u) & (atrib["funcao_origem"] == fn)]
    s, ac = sch.loc[a.index], autocorr.loc[a.index]
    e = est[(est["unidade"] == u) & (est["funcao_origem"] == fn)]
    _, forma = chaves_do_schema(est, u, fn)
    ped = s["pedido"].astype(str).value_counts()
    pedido, n_ped = ped.index[0], int(ped.iloc[0])
    por_posicao = pedido.lstrip("-").isdigit()
    acesso_errado = f"r[{pedido}]" if por_posicao else f"r['{pedido}']"
    trocas = ac[ac["tipo"] == "1 · troca de chave"]
    chave_nova, n_nova = Counter(k for ks in trocas["chaves_novas"].dropna() for k in ks.split(", ")).most_common(1)[0]
    cadeias, n_consertos = acesso_do_conserto(u, fn, atrib, eu, autocorr, passos)
    com_chave = [(k, v) for k, v in cadeias.most_common() if k.startswith(f"['{chave_nova}']")]
    acesso_certo, n_acesso = com_chave[0] if com_chave else (f"['{chave_nova}']", 0)
    p = presenca.xs((rotulo(u), fn), level=["unidade", "função"])
    n_prompts = int(p["de"].iloc[0])
    errada_no_bloco = (not por_posicao and pedido in p.index and p.loc[pedido, "tipo"].startswith("errada")
                       and int(p.loc[pedido, "no bloco da ferramenta"]) == n_prompts)
    reais_no_bloco = int((p.loc[p["tipo"] == "real", "no bloco da ferramenta"] > 0).sum())

    if por_posicao:
        descr = (f"`{fn}` devolve `{forma}`. O agente indexou o retorno por posição (`{acesso_errado}`), como se fosse "
                 f"lista, em {n_ped}/{len(s)} erros.")
    else:
        descr = f"`{fn}` devolve `{forma}`. O agente pediu a chave `'{pedido}'`, que não existe no retorno, em {n_ped}/{len(s)} erros."
    if errada_no_bloco:
        descr += f" É a chave que o bloco da ferramenta no system prompt declara ({n_prompts}/{n_prompts} prompts)."
    elif reais_no_bloco == 0:
        descr += " O bloco da ferramenta no system prompt não declara nenhuma chave do retorno."
    corr = f"Ao ler o retorno de `{fn}`, use `r{acesso_certo}`. Forma do retorno: `{forma}`."
    if errada_no_bloco:
        corr += f" O system prompt declara `'{pedido}'` para esta ferramenta; o retorno real não tem essa chave."

    loc = _alternando_meses(a, 10)
    st = status.xs((rotulo(u), fn))
    c = conf[(conf["unidade"] == rotulo(u)) & (conf["função"] == fn)]
    tb = tabela_estabilidade(e).reset_index()
    validation = {
        "passo2_leitura": f"literal_eval e regex concordam em {int(s['concordam'].fillna(False).astype(bool).sum())}/"
                          f"{int(s['concordam'].notna().sum())} objetos lidos",
        "passo3_consertos": f"{len(trocas)} trocas de chave, {int(trocas['bate_schema'].fillna(False).astype(bool).sum())} "
                            f"batem com o schema; '{chave_nova}' lida em {n_nova}; acesso `r{acesso_certo}` escrito pelo agente em "
                            f"{n_acesso}/{n_consertos} consertos de troca de chave com o step seguinte sem erro",
        "passo4_estabilidade": "; ".join(f"{x['nível (chaves de topo)']}: {x['veredito']} — meses {x['meses observados']}"
                                         for _, x in tb.iterrows()),
        "passo5_cobertura": f"{st['ocorrências lidas/confirmadas']}/{st['ocorrências']} ocorrências lidas ou confirmadas "
                            f"({st['cobertura']}, {st['confirmados por AttributeError']} confirmadas por AttributeError); "
                            f"cobertura estrita: {st['cobertura estrita']}; status na leitura estrita: {st['status na leitura estrita']}",
        "passo6_amostra": f"{len(c)} casos sorteados (semente {SEMENTE_P6}); conferência automática — todas as chaves no log "
                          f"de print: {int(c['(2) chaves do schema no log de print'].str.startswith('sim').sum())}/{len(c)}; "
                          "verificação humana: pendente",
        "prompt_11_3": (f"o bloco da ferramenta declara a chave errada '{pedido}'" if errada_no_bloco
                        else "o bloco da ferramenta não declara nenhuma chave do retorno" if reais_no_bloco == 0
                        else f"o bloco da ferramenta declara {reais_no_bloco} chave(s) real(is)"),
        "conteudo_escrito_a_mao_que_este_registro_substitui": UNI[u][2],
    }
    if errada_no_bloco:
        validation["destino"] = ("em aberto — a correção contradiz o system prompt: memória × não-memória · harness "
                                 "(07-relatorio-mineracao-unidades-n2-n10.md §6.1)")
    elif reais_no_bloco == 0:
        validation["destino"] = ("memória — o prompt não declara nenhuma chave do retorno desta ferramenta; sem "
                                 "contradição de contrato, é lacuna de informação (07-relatorio-mineracao-unidades-n2-n10.md §6.1)")
    else:
        validation["destino"] = ("memória — o prompt declara parte do schema real; sem contradição de contrato "
                                 "(07-relatorio-mineracao-unidades-n2-n10.md §6.1)")
    validation["analise_funda"] = "pendente"
    impact = None
    # §11.9: quando existe uma análise funda desta candidata (payload entregue × schema real), ela decide
    # destino/impact por regra fixada antes de rodar (06-racionais-mineracao-unidades-n2-n10.md §9) — substitui a frase genérica acima.
    # Campo renomeado de `passo_11_9` para `analise_funda` (17/09) na padronização de schema entre unidades —
    # mesmo conteúdo; nome genérico porque a análise funda de outra unidade não é necessariamente a §11.9.
    if desfecho and (u, fn) in desfecho:
        d = desfecho[(u, fn)]
        # Chaves opcionais (17/09): uma análise funda pode só documentar (nota), sem decidir destino/impact —
        # caso da nº2 (§11.10), diferente da nº10 (§11.9), que decide os três.
        if "destino" in d:
            validation["destino"] = d["destino"]
        if "nota" in d:
            validation["analise_funda"] = d["nota"]
        if "impact" in d:
            impact = d["impact"]
    return {
        "category": f"{UNIDADES[u]} · {UNI[u][0]} ({UNI[u][1]})",
        "scope": {"role": ", ".join(sorted(a["role"].unique())), "tool": fn},
        "location": {"total_erros": len(a),
                     "casos": [{"exec_id": r["exec_id"], "role": r["role"], "idx": int(r["idx"]), "mes": r["mes"]} for r in loc]},
        "evidence": [{"exec_id": r["exec_id"], "role": r["role"], "idx": int(r["idx"]), "mes": r["mes"],
                      "pedido": str(sch.loc[r.name, "pedido"]), "forma_do_objeto_indexado": sch.loc[r.name, "forma"]}
                     for r in loc[:3]],
        "impact": impact,
        "description": descr,
        "correction_guidance": corr,
        "occurrences": {"erros": len(a), "ocorrencias": int(a["ocorrencia"].nunique()),
                        "execucoes": int(a["exec_id"].nunique()), "meses": [str(m) for m in sorted(a["mes"].unique())]},
        "status": st["status"],
        "validation": validation,
    }
# ───────────────────────── §11.9 — leituras de validar_quebra_sigilo, payload entregue ─────────────────────────
# Pré-registro: 06-racionais-mineracao-unidades-n2-n10.md §9, "Análise funda da nº10". Emenda de cadeia e a medida direta pelo payload:
# mesma seção, "Execução da análise funda" — corrigido depois que a primeira rodada (fora do notebook) tratou
# `.get(a, .get(b))` como duas leituras independentes em vez de uma cadeia só.
FALHOU_QS = re.compile(r"^Code execution failed at line '(.*?)' due to: ", re.S)


def linha_da_falha(codigo, err):
    """A linha do step em que a execução parou, lida da própria mensagem de erro. None se não localizável —
    mesma lógica já usada dentro de resolve(), extraída aqui para reuso."""
    m = FALHOU_QS.search(err or "")
    if not m:
        return None
    primeira = m.group(1).strip().split("\n")[0].strip()
    try:
        arv = ast.parse(codigo)
    except SyntaxError:
        return None
    for n in ast.walk(arv):
        if isinstance(n, ast.stmt):
            seg = ast.get_source_segment(codigo, n) or ""
            if seg.strip().split("\n")[0].strip() == primeira:
                return n.lineno
    return None


def chamadas_de(codigo, fn):
    """[(var, linha)] para cada `x = fn(...)` no código do step."""
    try:
        arv = ast.parse(codigo)
    except SyntaxError:
        return []
    out = []
    for n in ast.walk(arv):
        if not isinstance(n, (ast.Assign, ast.AnnAssign)) or n.value is None:
            continue
        alvo = n.targets[0] if isinstance(n, ast.Assign) else n.target
        if isinstance(alvo, ast.Name) and origem(n.value) == ("call", fn):
            out.append((alvo.id, n.lineno))
    return out


def cadeia_get(call, var):
    """Uma chamada `<var>.get('k', <plano B>)` pode ter o plano B sendo outro `.get` sobre a mesma var —
    uma cadeia, não duas leituras. Devolve a lista de chaves pedidas nessa cadeia, na ordem em que resolvem."""
    chaves, n = [], call
    while (isinstance(n, ast.Call) and isinstance(n.func, ast.Attribute) and n.func.attr == "get"
           and isinstance(n.func.value, ast.Name) and n.func.value.id == var and n.args
           and isinstance(n.args[0], ast.Constant)):
        chaves.append(n.args[0].value)
        n = n.args[1] if len(n.args) > 1 else None
    return chaves


def resolver_nome_no_step(arv, nome, ate_linha):
    """A última atribuição de `nome` no step, antes de `ate_linha` — o nó do lado direito, ou None."""
    melhor = None
    for n in ast.walk(arv):
        if isinstance(n, ast.Assign) and n.value is not None and n.lineno < ate_linha and any(
                isinstance(t, ast.Name) and t.id == nome for t in n.targets):
            if melhor is None or n.lineno > melhor.lineno:
                melhor = n
    return melhor.value if melhor else None


def classificar_expr_leitura(v, chaves_reais, fn=None, arv=None, linha_ref=None, profundidade=0):
    """Como uma expressão que (talvez) lê o retorno da ferramenta resolve: indexação direta, cadeia de `.get`
    (resolve na 1ª chave real — não conta o plano B como leitura própria; emenda de 17/09,
    03-procedimento-validacao.md §1.10, depois que a 1ª rodada contava as duas pernas da cadeia como leituras
    independentes), objeto inteiro repassado, ou literal fixo. Quando a expressão é só um nome de variável,
    resolve um nível — acha onde essa variável foi montada no mesmo step (ex.: `x = r.get(...)` seguido de
    `{..., 'campo': x}`) e classifica pelo que a monta de verdade, em vez de marcar "objeto inteiro" à toa."""
    if isinstance(v, ast.Subscript) and isinstance(v.slice, ast.Constant):
        k = v.slice.value
        return ("(a) chave real" if k in chaves_reais else "(b) chave declarada, com erro", k, "indexação direta")
    if isinstance(v, ast.Call) and isinstance(v.func, ast.Attribute) and v.func.attr == "get" and v.args:
        chaves = cadeia_get(v, v.func.value.id if isinstance(v.func.value, ast.Name) else None)
        resolvida = next((k for k in chaves if k in chaves_reais), None)
        if resolvida is not None:
            return ("(a) chave real", resolvida, "cadeia .get, resolve na 1ª chave real" if len(chaves) > 1 else ".get direto")
        return ("(c) chave declarada, sem erro", chaves[-1] if chaves else None,
                "cadeia .get, nenhuma chave existe" if len(chaves) > 1 else ".get direto, sem plano B que resolva")
    if isinstance(v, ast.Call) and isinstance(v.func, ast.Name) and fn is not None and v.func.id == fn:
        return ("(d) objeto inteiro repassado", None, "atribuído direto da chamada, sem ler nenhuma chave")
    if isinstance(v, ast.Name):
        if arv is not None and linha_ref is not None and profundidade < 3:
            rhs = resolver_nome_no_step(arv, v.id, linha_ref)
            if rhs is not None:
                return classificar_expr_leitura(rhs, chaves_reais, fn, arv, getattr(rhs, "lineno", linha_ref), profundidade + 1)
        return ("(d) objeto inteiro repassado", None, "nome nu, sem subscrito nem .get")
    if isinstance(v, ast.Constant) and isinstance(v.value, str):
        return ("(e) literal fixo", None, "string fixa no código, não vem da ferramenta")
    return ("(?) outra forma", None, type(v).__name__)


def classificar_campo_final(passos, exec_id, role, campo, fn, chaves_reais):
    """Como o campo `campo` do payload final foi construído, procurando em TODOS os steps do papel (a construção
    do dict pode estar num step, o `final_answer` que o envia em outro) a última vez que `'<campo>': <expr>`
    aparece num dict literal — mesma regra de reatribuição já usada em toda a §11: quem constrói por último vence
    (ex.: `2bb6ea3a…` lê a chave certa no idx 4 e um step depois, idx 5, sobrescreve com texto fixo — o que conta
    é o que sobrou no fim). Resolve um nível de indireção de variável (`resolver_nome_no_step`)."""
    ps = passos.get((exec_id, role), {})
    achados = []
    for idx in sorted(ps):
        try:
            arv = ast.parse(ps[idx]["code"])
        except SyntaxError:
            continue
        for n in ast.walk(arv):
            if not isinstance(n, ast.Dict):
                continue
            for k, v in zip(n.keys, n.values):
                if isinstance(k, ast.Constant) and k.value == campo:
                    linha = getattr(v, "lineno", 0)
                    achados.append((idx, linha, classificar_expr_leitura(v, chaves_reais, fn, arv, linha)))
    if not achados:
        return None
    return max(achados, key=lambda a: (a[0], a[1]))[2]


def categoria_valor_entregue(v):
    """O valor de request.quebra_sigilo (ou campo análogo) categorizado sem nunca imprimir texto de caso."""
    if v is None:
        return "ausente (None)"
    if isinstance(v, dict):
        return f"objeto inteiro ({', '.join(sorted(chave_segura(k) for k in v))})"
    if isinstance(v, list):
        return f"<lista, {len(v)} itens>"
    if isinstance(v, str):
        if v == "":
            return "string vazia"
        n = norm_acento(v)
        if n in ("SIM", "NAO") and len(v) <= 4:
            return f"'{v}' (valor esperado)"
        if len(v) <= 12 and not re.search(r"\d{5,}", v):
            return f"'{v}' (curto, inesperado)"
        return f"<texto livre, {len(v)} chars>"
    return f"<{type(v).__name__}>"


def norm_acento(s):
    return "".join(c for c in unicodedata.normalize("NFD", str(s)) if unicodedata.category(c) != "Mn").upper()


def payload_entregue_qs(passos, exec_id, role, chave="quebra_sigilo"):
    """A ÚLTIMA entrega desta execução com `chave` dentro de action_output['request'] — medida direta, não
    inferência de código. None se a execução nunca chegou a entregar a chave."""
    achado = None
    for idx in sorted(passos.get((exec_id, role), {})):
        st = passos[(exec_id, role)][idx]
        out = st.get("out")
        if isinstance(out, dict) and isinstance(out.get("request"), dict) and chave in out["request"]:
            achado = out["request"][chave]
    return achado


def base_deriva_de(v, retorno_vars, arv, linha_ref, profundidade=0):
    """True se a expressão `v` (o lado não-literal de uma comparação) vem, direta ou indiretamente (um nível,
    dentro do mesmo step), de alguma variável em `retorno_vars` — as que de fato receberam a chamada da
    ferramenta. Sem isso, qualquer comparação de string em QUALQUER lugar do step contaria, mesmo sobre uma
    variável sem nenhuma relação com o retorno da ferramenta (achado do próprio teste desta célula: uma
    comparação sobre `new_validade`, de outro campo qualquer, seria contada à toa)."""
    if isinstance(v, ast.Name):
        if v.id in retorno_vars:
            return True
        if profundidade < 3:
            rhs = resolver_nome_no_step(arv, v.id, linha_ref)
            if rhs is not None:
                return base_deriva_de(rhs, retorno_vars, arv, getattr(rhs, "lineno", linha_ref), profundidade + 1)
        return False
    if isinstance(v, (ast.Subscript, ast.Attribute)):
        return base_deriva_de(v.value, retorno_vars, arv, linha_ref, profundidade)
    if isinstance(v, ast.Call) and isinstance(v.func, ast.Attribute):
        return base_deriva_de(v.func.value, retorno_vars, arv, linha_ref, profundidade)
    return False


def comparacoes_grafia_qs(passos, exec_id, role, fn, valores_reais_norm):
    """Comparações (==, !=, in) de uma variável DERIVADA DO RETORNO DA FERRAMENTA (`base_deriva_de`) com um
    literal de string que NÃO está entre os valores observados no trace, mas vira um deles ao normalizar
    caixa/acento — a régua de grafia do pré-registro (06-racionais-mineracao-unidades-n2-n10.md §9: a referência é o trace, nunca o
    prompt). Para cada uma, verifica se a linha da comparação está antes, na, ou depois da linha em que o step
    falhou (`linha_da_falha`) — só "antes" quer dizer que a comparação chegou a ser avaliada."""
    retorno_vars = {var for idx in passos.get((exec_id, role), {})
                    for var, _ in chamadas_de(passos[(exec_id, role)][idx]["code"], fn)}
    out = []
    for idx in sorted(passos.get((exec_id, role), {})):
        st = passos[(exec_id, role)][idx]
        try:
            arv = ast.parse(st["code"])
        except SyntaxError:
            continue
        lf = linha_da_falha(st["code"], st["err_msg"])
        for n in ast.walk(arv):
            if not isinstance(n, ast.Compare):
                continue
            if not base_deriva_de(n.left, retorno_vars, arv, n.lineno):
                continue
            for op, comp in zip(n.ops, n.comparators):
                if not isinstance(op, (ast.Eq, ast.NotEq, ast.In, ast.NotIn)):
                    continue
                for x in ast.walk(comp):
                    if not (isinstance(x, ast.Constant) and isinstance(x.value, str) and len(x.value) <= 8):
                        continue
                    lit = x.value
                    if lit in valores_reais_norm or norm_acento(lit) not in valores_reais_norm:
                        continue  # bate igual, ou não é variante de nenhum valor observado
                    linha = n.lineno
                    alcance = ("não avaliada — step falhou antes" if lf is not None and linha > lf
                               else "indeterminada — mesma linha da falha" if lf is not None and linha == lf
                               else "avaliada")
                    out.append({"exec_id": exec_id, "role": role, "idx": idx, "mes": st["mes"],
                               "literal": lit, "alcance": alcance})
    return out




### 11.1 · Passo 1 — de qual ferramenta vem cada erro

**O que faz.** Para cada erro das duas unidades, acha no comando que falhou o acesso que quebrou e segue a variável
até a chamada que a criou — no mesmo comando, no mesmo step ou num step anterior do mesmo papel.

**Como ler.** A tabela tem uma linha por `(unidade, função de origem)`, com erros, ocorrências, execuções e meses, e o
**destino** de cada sub-unidade (candidata / documentar e monitorar / atribuição a refazer). Abaixo dela, o veredito da
régua: **(a)** uma função com ≥90% → uma ferramenta só; **(b)** outra função com ≥10% → a unidade se divide.


In [3]:
# 11.1 · Passo 1 — de qual ferramenta vem cada erro
STEPS_P1 = indexar_steps(RAW)
ATRIB = atribuir_origem(EU, STEPS_P1, list(UNIDADES))
SUB_P1 = sub_unidades(ATRIB, MIN_EXECS, MIN_MESES)
display(tabela_sub_unidades(SUB_P1))

VEREDITO_P1 = {}
for u in UNIDADES:
    for nivel in ("erros", "ocorrências"):
        VEREDITO_P1[(u, nivel)], texto = veredito(ATRIB, u, nivel)
        print(texto)

BUCKET_CONSULTA = bucket_consulta(ATRIB, SUB_P1)
print(f"bucket de consulta (não passa na triagem e cobre <10% da unidade): {len(BUCKET_CONSULTA)} erros")
if len(BUCKET_CONSULTA):
    display(BUCKET_CONSULTA)

# evidência (ver 11.0): todos os erros fora da função dominante + o primeiro caso de cada `via` da dominante
dominante = ATRIB.groupby("unidade")["funcao_origem"].agg(lambda s: s.value_counts().index[0])
eh_dom = ATRIB["funcao_origem"].eq(ATRIB["unidade"].map(dominante))
registrar_evidencia(
    "11.1_origem_do_erro", "§11.1",
    "a função de origem atribuída a cada erro das unidades nº2 e nº10 — em especial os erros fora da função dominante, "
    "que decidem as regras (a) e (b).",
    "todos os erros atribuídos fora da função dominante da unidade, mais o primeiro caso, na ordem (mês, exec_id, idx), "
    "de cada `via` de atribuição da função dominante.",
    pd.concat([ATRIB[~eh_dom].assign(motivo="fora da função dominante da unidade"),
               primeiro_por(ATRIB[eh_dom], ["unidade", "via"]).assign(
                   motivo=lambda d: "função dominante, primeiro caso com via: " + d["via"])])
      .assign(ferramenta=lambda d: d["funcao_origem"])[["exec_id", "role", "idx", "mes", "unidade", "ferramenta", "via", "motivo"]],
    {"atribuicao_por_erro": ATRIB})


erros  ocorrencias  \
unidade                    função                                        
nº2 · U_contrato_dict      get_available_documents     91           86   
                           não resolvido                4            1   
nº10 · U_campo_inexistente validar_quebra_sigilo        7            7   
                           extrair_evidencias           1            1   
                           get_available_documents      1            1   
                           não resolvido                1            1   

                                                    execucoes  meses  \
unidade                    função                                      
nº2 · U_contrato_dict      get_available_documents         86      6   
                           não resolvido                    1      1   
nº10 · U_campo_inexistente validar_quebra_sigilo            7      3   
                           extrair_evidencias               1      1   
                           get_available_documents          1      1   
                           não resolvido                    1      1   

                                                               papeis  \
unidade                    função                                       
nº2 · U_contrato_dict      get_available_documents  ConversationAgent   
                           não resolvido            ConversationAgent   
nº10 · U_campo_inexistente validar_quebra_sigilo        RespostaBacen   
                           extrair_evidencias           RespostaBacen   
                           get_available_documents  ConversationAgent   
                           não resolvido            ConversationAgent   

                                                   passa triagem  \
unidade                    função                                  
nº2 · U_contrato_dict      get_available_documents           sim   
                           não resolvido                       —   
nº10 · U_campo_inexistente validar_quebra_sigilo             sim   
                           extrair_evidencias                não   
                           get_available_documents           não   
                           não resolvido                       —   

                                                    % ocorr. na unidade  \
unidade                    função                                         
nº2 · U_contrato_dict      get_available_documents                 98.9   
                           não resolvido                            1.1   
nº10 · U_campo_inexistente validar_quebra_sigilo                   70.0   
                           extrair_evidencias                      10.0   
                           get_available_documents                 10.0   
                           não resolvido                           10.0   

                                                                                 destino  
unidade                    função                                                         
nº2 · U_contrato_dict      get_available_documents                             candidata  
                           não resolvido               documentar — atribuição a refazer  
nº10 · U_campo_inexistente validar_quebra_sigilo                               candidata  
                           extrair_evidencias       documentar e monitorar (2ª extração)  
                           get_available_documents  documentar e monitorar (2ª extração)  
                           não resolvido               documentar — atribuição a refazer

nº2 · U_contrato_dict [erros]: get_available_documents = 91/96 (94.8%) → (a) uma ferramenta só (≥90%)
nº2 · U_contrato_dict [ocorrências]: get_available_documents = 86/87 (98.9%) → (a) uma ferramenta só (≥90%)
nº10 · U_campo_inexistente [erros]: validar_quebra_sigilo = 7/10 (70.0%) → (b) divide — outra função com ≥10%: extrair_evidencias, get_available_documents
nº10 · U_campo_inexistente [ocorrências]: validar_quebra_sigilo = 7/10 (70.0%) → (b) divide — outra função com ≥10%: extrair_evidencias, get_available_documents
bucket de consulta (não passa na triagem e cobre <10% da unidade): 0 erros
evidência: 12 casos e 1 tabela(s) em resultados/evidencia/11.1_origem_do_erro/ — crus e visões por caso: `python drill_down.py evidencia 11.1_origem_do_erro`


### 11.2 · Passo 2 — o que a ferramenta devolve de verdade

**O que faz.** Quando o acesso falha, o smolagents imprime na mensagem (`Could not index {objeto} with '{chave}'`) o
objeto que o agente tentou indexar. A célula lê esse objeto por dois métodos independentes — `ast.literal_eval` e um
regex só de nomes de chave — e descreve a sua **forma** (chaves e tipos).

**Como ler.** A primeira tabela é a leitura: quantas mensagens trazem o objeto, quantas cada método leu, se os dois
concordam, e a sanidade — a chave pedida **não** pode existir no objeto, senão não teria havido erro. A segunda é o
resultado: a forma do objeto e o que o agente pediu a ele. O objeto pode ser o retorno inteiro ou um pedaço dele (um
documento de dentro da lista, por exemplo).


In [4]:
# 11.2 · Passo 2 — o que a ferramenta devolve de verdade
SCHEMA = ler_schema(ATRIB, EU, STEPS_P1)
CHAVES_REAIS = chaves_reais(SCHEMA)
with pd.option_context("display.max_colwidth", None):
    display(tabela_leitura(SCHEMA))
    display(tabela_forma(SCHEMA))

# evidência (ver 11.0): o primeiro caso de cada forma distinta do objeto indexado
SCH = SCHEMA.assign(role=ATRIB["role"].values, ocorrencia=ATRIB["ocorrencia"].values)
registrar_evidencia(
    "11.2_schema_real", "§11.2",
    "a forma (chaves e tipos) do objeto que a mensagem de erro imprime, por unidade e função — e que a chave pedida pelo "
    "agente não existe nele.",
    "o primeiro caso, na ordem (mês, exec_id, idx), de cada forma distinta por unidade e função, inclusive "
    "'objeto não lido da mensagem'.",
    primeiro_por(SCH.assign(forma=SCH["forma"].fillna("(objeto não lido da mensagem)")), ["unidade", "funcao_origem", "forma"])
      .assign(ferramenta=lambda d: d["funcao_origem"], chaves=lambda d: d["chaves_topo"].map(lambda k: ";".join(k) if k else ""),
              motivo="primeiro caso desta forma do objeto indexado")
      [["exec_id", "role", "idx", "mes", "unidade", "ferramenta", "pedido", "forma", "chaves", "motivo"]],
    {"schema_por_erro": SCH})


erros objeto na mensagem  \
unidade                    função                                              
nº2 · U_contrato_dict      get_available_documents     91              89/91   
                           não resolvido                4                0/4   
                           meta_map                     1                0/1   
nº10 · U_campo_inexistente validar_quebra_sigilo        7                7/7   
                           extrair_evidencias           1                1/1   
                           get_available_documents      1                1/1   
                           não resolvido                1                1/1   

                                                   literal_eval lê regex lê  \
unidade                    função                                             
nº2 · U_contrato_dict      get_available_documents           89/89    89/89   
                           não resolvido                         —        —   
                           meta_map                              —        —   
nº10 · U_campo_inexistente validar_quebra_sigilo               7/7      7/7   
                           extrair_evidencias                  1/1      1/1   
                           get_available_documents             1/1      1/1   
                           não resolvido                       0/1      0/1   

                                                   os dois concordam  \
unidade                    função                                      
nº2 · U_contrato_dict      get_available_documents             89/89   
                           não resolvido                           —   
                           meta_map                                —   
nº10 · U_campo_inexistente validar_quebra_sigilo                 7/7   
                           extrair_evidencias                    1/1   
                           get_available_documents               1/1   
                           não resolvido                           —   

                                                   chave pedida existe no objeto  \
unidade                    função                                                  
nº2 · U_contrato_dict      get_available_documents                          0/88   
                           não resolvido                                       —   
                           meta_map                                            —   
nº10 · U_campo_inexistente validar_quebra_sigilo                             0/7   
                           extrair_evidencias                                0/1   
                           get_available_documents                           0/1   
                           não resolvido                                       —   

                                                   thought cita a chave pedida  \
unidade                    função                                                
nº2 · U_contrato_dict      get_available_documents                           —   
                           não resolvido                                     —   
                           meta_map                                          —   
nº10 · U_campo_inexistente validar_quebra_sigilo                           7/7   
                           extrair_evidencias                              1/1   
                           get_available_documents                         1/1   
                           não resolvido                                     —   

                                                   thought cita a função  \
unidade                    função                                          
nº2 · U_contrato_dict      get_available_documents                  5/91   
                           não resolvido                             0/4   
                           meta_map                                  1/1   
nº10 · U_campo_inexistente validar_quebra_sigilo                     6/7   
                  

erros
unidade                    função                  forma do objeto indexado                                                                                                                                                                pedido do agente             
nº2 · U_contrato_dict      get_available_documents {result: [[{hashDocumento: str, metadado: [{nomeMetadado: str, valorMetadado: str}], tipoExtracaoOcr: str}] | {<campo>: {<valor>: int}}]}                                               0                          85
                                                   (objeto não lido da mensagem)                                                                                                                                                           <attr>                      2
                                                   {hashDocumento: str, metadado: [{nomeMetadado: str, valorMetadado: str}], tipoExtracaoOcr: str}                                                                                         0                           2
                                                                                                                                                                                                                                           <slice>                     1
                                                   {result: [[{hashDocumento: str, iuDocsId: NoneType, iuDocsTenantId: NoneType, metadado: [{nomeMetadado: str, valorMetadado: str}], tipoExtracaoOcr: str}] | {<campo>: {<valor>: int}}]} 0                           1
                           não resolvido           (objeto não lido da mensagem)                                                                                                                                                           <attr>                      4
                           meta_map                (objeto não lido da mensagem)                                                                                                                                                           <attr>                      1
nº10 · U_campo_inexistente validar_quebra_sigilo   {justificativa: str, vazamento_sigilo: str}                                                                                                                                             quebra_sigilo               7
                           extrair_evidencias      {dados_evidencias: [{data_inicio_atraso: str | NoneType, final_cartao: str, nome_contrato: str, numero_contrato: str}]}                                                                 informacoes_evidencias      1
                           get_available_documents {nomeMetadado: str, valorMetadado: str}                                                                                                                                                 nom_docm_juri_mode          1
                           não resolvido           (objeto não lido da mensagem)                                                                                                                                                           <colunas>                   1

evidência: 10 casos e 1 tabela(s) em resultados/evidencia/11.2_schema_real/ — crus e visões por caso: `python drill_down.py evidencia 11.2_schema_real`


### 11.3 · Addendum ao Passo 2 — o que o system prompt diz que a ferramenta devolve

**O que faz.** Lê, no system prompt de cada step com erro, o bloco em que a ferramenta é declarada
(`def ferramenta(...)`), e conta por presença literal de texto se as chaves reais (Passo 2) e a chave errada que o
agente pediu aparecem no prompt.

**Como ler.** A primeira tabela é o texto literal do prompt sobre o retorno de cada ferramenta — é documentação da
ferramenta, não dado de caso — para comparar com a forma real do 11.2. A segunda tem uma linha por chave; a coluna que
responde a pergunta é **"como chave (':')"**, e "no bloco da ferramenta" diz se a chave está declarada para esta
ferramenta e não para outra. As colunas "palavra solta" e "entre aspas" mostram quanto uma busca frouxa enganaria
(prosa, valor de exemplo). A primeira tabela e a última coluna não estavam no pré-registro (ver `03` §1.7).


In [5]:
# 11.3 · Addendum ao Passo 2 — o que o system prompt diz que a ferramenta devolve
PRESENCA = presenca_no_prompt(ATRIB, EU, STEPS_P1, CHAVES_REAIS)
with pd.option_context("display.max_colwidth", None):
    display(tabela_retorno_declarado(ATRIB, STEPS_P1, CHAVES_REAIS))
    display(PRESENCA)

# evidência (ver 11.0): o primeiro erro de cada ferramenta com schema lido — o system prompt enviado ao agente está no cru
com_schema = ATRIB[ATRIB["funcao_origem"].map(lambda f: bool(CHAVES_REAIS.get(f)))]
registrar_evidencia(
    "11.3_prompt_declara", "§11.3",
    "o que o bloco `def ferramenta(...)` do system prompt diz que cada ferramenta devolve, e se as chaves reais e a chave "
    "pedida pelo agente aparecem nele.",
    "o primeiro erro, na ordem (mês, exec_id, idx), de cada ferramenta cujo schema foi lido no 11.2. O bloco é o mesmo em "
    "todos os erros de cada ferramenta; `drill_down.py ferramenta <nome>` mostra as variantes no trace inteiro.",
    primeiro_por(com_schema, ["funcao_origem"])
      .assign(ferramenta=lambda d: d["funcao_origem"],
              chaves=lambda d: d["funcao_origem"].map(lambda f: ";".join(sorted(CHAVES_REAIS[f]))),
              motivo="primeiro erro desta ferramenta (o system prompt enviado ao agente está no cru)")
      [["exec_id", "role", "idx", "mes", "unidade", "ferramenta", "chaves", "motivo"]],
    {"presenca_no_prompt": PRESENCA.reset_index(),
     "retorno_declarado": tabela_retorno_declarado(ATRIB, STEPS_P1, CHAVES_REAIS).reset_index()})


,erros (nas duas unidades),variantes do bloco nesses erros,assinatura no prompt,retorno declarado no prompt
função,,,,
get_available_documents,92,1,"def get_available_documents(filter_field_name: string, filter_field_value: string, domains: array) -> object:","This is a tool that returns a list of documents and a summary of the kinds of documents from a list of dominios, to use it you must first choose a list of valid dominios and the correct filter_field_name from the available ones. The tool may return a very large amount of data, use the summary provided in the response to understand the data in the documents and filter accordingly."
validar_quebra_sigilo,7,1,"def validar_quebra_sigilo(reclamacao: string, evidencias: array, resposta: string) -> object:","json com a validação de quebra de sigilo no formato: {""quebra_sigilo"": ""SIM"" ou ""NÃO"", ""motivo"": ""justificativa""}"
extrair_evidencias,1,1,"def extrair_evidencias(evidencias: array, assunto: string) -> object:","json com as informações extraídas das evidências, no seguinte formato: { ""informacoes_evidencias"": [<lista de informações extraídas de cada evidência>] }"


tipo  \
unidade                    função                  chave                                                 
nº2 · U_contrato_dict      get_available_documents hashDocumento                                  real   
                                                   iuDocsId                                       real   
                                                   iuDocsTenantId                                 real   
                                                   metadado                                       real   
                                                   nomeMetadado                                   real   
                                                   result                                         real   
                                                   tipoExtracaoOcr                                real   
                                                   valorMetadado                                  real   
nº10 · U_campo_inexistente validar_quebra_sigilo   justificativa                                  real   
                                                   vazamento_sigilo                               real   
                                                   quebra_sigilo           errada (pedida pelo agente)   
                           get_available_documents hashDocumento                                  real   
                                                   iuDocsId                                       real   
                                                   iuDocsTenantId                                 real   
                                                   metadado                                       real   
                                                   nomeMetadado                                   real   
                                                   result                                         real   
                                                   tipoExtracaoOcr                                real   
                                                   valorMetadado                                  real   
                                                   nom_docm_juri_mode      errada (pedida pelo agente)   
                           extrair_evidencias      dados_evidencias                               real   
                                                   data_inicio_atraso                             real   
                                                   final_cartao                                   real   
                                                   nome_contrato                                  real   
                                                   numero_contrato                                real   
                                                   informacoes_evidencias  errada (pedida pelo agente)   

                                                                           palavra solta  \
unidade                    função                  chave                                   
nº2 · U_contrato_dict      get_available_documents hashDocumento                      91   
                                                   iuDocsId                            0   
                                                   iuDocsTenantId                      0   
                                                   metadado                            0   
                                                   nomeMetadado                       91   
                                                   result                             91   
                                                   tipoExtracaoOcr                    91   
                                                   valorMetadado                       0   
nº10 · U_campo_inexistente validar_quebra_sigilo   justificativa                       7   
                                                   vazamento_sigilo                    0   
                         

evidência: 3 casos e 2 tabela(s) em resultados/evidencia/11.3_prompt_declara/ — crus e visões por caso: `python drill_down.py evidencia 11.3_prompt_declara`


### 11.4 · Passo 3 — o que o agente fez no step seguinte

**O que faz.** Para cada erro, lê o código do step seguinte do mesmo papel e classifica o que aconteceu com a
variável que quebrou: **(1) troca de chave** — passa a ler por outra chave, conferida contra a forma do 11.2 (marcada
"com guarda de tipo" quando o acesso antigo fica num ramo `if`); **(2) conserto silencioso** — a mesma chave errada,
protegida por `.get(chave, padrão)` ou `try/except`; **(3) sem conserto comparável**. Repetir o acesso errado fica fora
dos três; erro de atributo ou de colunas é "não aplicável".

**Como ler.** A primeira tabela conta os desfechos por ferramenta e, nas trocas, quantas batem com a forma real e
quantas deixam o step seguinte sem erro. A segunda dá o detalhe: a chave nova lida, ou o segundo parâmetro do `.get`.
Nomes de variável não são impressos (podem embutir identificadores).


In [6]:
# 11.4 · Passo 3 — o que o agente fez no step seguinte
AUTOCORR = classificar_conserto(ATRIB, EU, STEPS_P1, CHAVES_REAIS)
display(tabela_desfechos(AUTOCORR))
display(tabela_detalhe(AUTOCORR))

# evidência (ver 11.0): por ferramenta, o primeiro caso de cada desfecho de correção — os cinco logs de 03 §1.8
AC = AUTOCORR.drop(columns="variavel").assign(mes=ATRIB["mes"].values)
AC["desfecho_de_correcao"] = np.select(
    [AC["detalhe"].str.startswith("com guarda", na=False), AC["tipo"].eq("1 · troca de chave"), AC["tipo"].eq("2 · conserto silencioso")],
    ["troca com guarda de tipo", "troca sem guarda", "conserto com .get"], default="")
registrar_evidencia(
    "11.4_conserto", "§11.4",
    "o que o agente fez no step seguinte ao erro — e, junto com o 11.3, que o prompt declara um contrato, a ferramenta "
    "devolve outro, e o agente descobre a chave real na resposta ao step que falhou (`03-procedimento-validacao.md` §1.8).",
    "por ferramenta, o primeiro caso, na ordem (mês, exec_id, idx), de cada desfecho de correção do Passo 3 — troca sem "
    "guarda, troca com guarda de tipo, conserto com `.get` —, no máximo dois por ferramenta.",
    primeiro_por(AC[AC["desfecho_de_correcao"] != ""], ["funcao_origem", "desfecho_de_correcao"]).groupby("funcao_origem").head(2)
      .assign(ferramenta=lambda d: d["funcao_origem"],
              chaves=lambda d: SCH.loc[d.index, "chaves_topo"].map(lambda k: ";".join(k) if k else ""),
              motivo=lambda d: "primeiro caso do desfecho: " + d["desfecho_de_correcao"])
      [["exec_id", "role", "idx", "mes", "unidade", "ferramenta", "pedido", "desfecho_de_correcao", "chaves_novas", "chaves", "motivo"]],
    {"conserto_por_erro": AC})


erros  \
unidade                    função                           
nº2 · U_contrato_dict      get_available_documents     91   
                           não resolvido                4   
                           meta_map                     1   
nº10 · U_campo_inexistente validar_quebra_sigilo        7   
                           extrair_evidencias           1   
                           get_available_documents      1   
                           não resolvido                1   

                                                    1 · troca de chave (sem guarda)  \
unidade                    função                                                     
nº2 · U_contrato_dict      get_available_documents                               64   
                           não resolvido                                          0   
                           meta_map                                               0   
nº10 · U_campo_inexistente validar_quebra_sigilo                                  6   
                           extrair_evidencias                                     1   
                           get_available_documents                                1   
                           não resolvido                                          0   

                                                    1 · troca de chave (com guarda de tipo)  \
unidade                    função                                                             
nº2 · U_contrato_dict      get_available_documents                                       17   
                           não resolvido                                                  0   
                           meta_map                                                       0   
nº10 · U_campo_inexistente validar_quebra_sigilo                                          0   
                           extrair_evidencias                                             0   
                           get_available_documents                                        0   
                           não resolvido                                                  0   

                                                    2 · conserto silencioso  \
unidade                    função                                             
nº2 · U_contrato_dict      get_available_documents                        0   
                           não resolvido                                  0   
                           meta_map                                       0   
nº10 · U_campo_inexistente validar_quebra_sigilo                          1   
                           extrair_evidencias                             0   
                           get_available_documents                        0   
                           não resolvido                                  0   

                                                    3 · sem conserto comparável  \
unidade                    função                                                 
nº2 · U_contrato_dict      get_available_documents                            7   
                           não resolvido                                      0   
                           meta_map                                           0   
nº10 · U_campo_inexistente validar_quebra_sigilo                              0   
                           extrair_evidencias                                 0   
                           get_available_documents                            0   
                           não resolvido                                      0   

                                                    fora dos três tipos · repetiu o acesso errado  \
unidade                    função                                                                   
nº2 · U_contrato_dict      get_available_documents                                              1   
                           não resolvido                                                        0  

erros
unidade                    função                  desfecho                                      detalhe                                            chave nova / 2º parâmetro do .get                      
nº2 · U_contrato_dict      get_available_documents 1 · troca de chave                            confirma o schema da ferramenta (Passo 2)          result                                               64
                                                                                                 com guarda de tipo — o acesso antigo fica num r... result                                               16
                                                   3 · sem conserto comparável                   step seguinte não lê mais a variável               —                                                     6
                                                   não aplicável                                 erro de attr, não de chave/índice                  —                                                     2
                                                   1 · troca de chave                            com guarda de tipo — o acesso antigo fica num r... hashDocumento, metadado, result, tipoExtracaoOcr      1
                                                   3 · sem conserto comparável                   variável indexada não identificada                 —                                                     1
                                                   fora dos três tipos · repetiu o acesso errado mesma chave, sem proteção nem guarda               —                                                     1
                           não resolvido           não aplicável                                 erro de attr, não de chave/índice                  —                                                     4
                           meta_map                não aplicável                                 erro de attr, não de chave/índice                  —                                                     1
nº10 · U_campo_inexistente validar_quebra_sigilo   1 · troca de chave                            confirma o schema da ferramenta (Passo 2)          vazamento_sigilo                                      5
                                                                                                                                                    justificativa, vazamento_sigilo                       1
                                                   2 · conserto silencioso                       .get                                               <var>.get('vazamento_sigilo')                         1
                           extrair_evidencias      1 · troca de chave                            confirma o schema da ferramenta (Passo 2)          dados_evidencias                                      1
                           get_available_documents 1 · troca de chave                            confirma o schema da ferramenta (Passo 2)          hashDocumento, tipoExtracaoOcr                        1
                           não resolvido           não aplicável                                 erro de colunas, não de chave/índice               —                                                     1

evidência: 5 casos e 1 tabela(s) em resultados/evidencia/11.4_conserto/ — crus e visões por caso: `python drill_down.py evidencia 11.4_conserto`


### 11.5 · Passo 4 — o schema é o mesmo em todos os meses?

**O que faz.** Para cada sub-unidade candidata do Passo 1, compara a forma do objeto lido no Passo 2 entre erros e entre
meses, separando os níveis da estrutura pelas chaves de topo (o retorno inteiro × um documento de dentro dele). Cada
forma é comparada com a mais comum do seu nível: **idêntica**; **campos a mais** (a forma comum inteira está lá, com os
mesmos tipos, mais alguma chave); ou **contradição** (falta chave da forma comum, ou um tipo muda).

**Como ler.** A primeira tabela conta erros por mês, nível e comparação. A segunda dá o veredito por nível — "estável",
"estável, com campos a mais em <meses>" ou "schema mudou em <meses>" — e, ao lado, a leitura estrita, em que qualquer
diferença conta. Só os meses com erro são observados: mês sem erro não diz nada sobre o schema.


In [7]:
# 11.5 · Passo 4 — o schema é o mesmo em todos os meses?
CANDIDATAS = sorted((tuple(i) for i in SUB_P1.index[SUB_P1["destino"] == "candidata"]), key=lambda k: list(UNIDADES).index(k[0]))
EST = estabilidade(SCH, CANDIDATAS)
display(tabela_meses(EST))
with pd.option_context("display.max_colwidth", None):
    display(tabela_estabilidade(EST))

# evidência (ver 11.0): o primeiro caso de cada mês em cada nível + todos os casos cuja forma difere da comum
registrar_evidencia(
    "11.5_estabilidade", "§11.5",
    "a forma do retorno de cada ferramenta candidata é a mesma em todos os meses com erro, por nível da estrutura — e, "
    "onde difere, como.",
    "o primeiro caso, na ordem (mês, exec_id, idx), de cada mês em cada nível, mais todos os casos cuja forma difere da comum.",
    pd.concat([primeiro_por(EST, ["unidade", "funcao_origem", "nivel", "mes"]).assign(motivo="primeiro caso do mês neste nível"),
               EST[EST["comparacao"] != "idêntica"].assign(motivo=lambda d: "forma difere da comum: " + d["comparacao"])])
      .assign(ferramenta=lambda d: d["funcao_origem"], chaves=lambda d: d["chaves_topo"].map(";".join))
      [["exec_id", "role", "idx", "mes", "unidade", "ferramenta", "nivel", "forma", "forma_comum", "comparacao", "chaves", "motivo"]],
    {"estabilidade_por_erro": EST, "estabilidade": tabela_estabilidade(EST).reset_index()})


erros  \
unidade                    função                  nível (chaves de topo)                     comparação com a forma comum          
nº2 · U_contrato_dict      get_available_documents {result}                                   idêntica                         85   
                                                   {hashDocumento, metadado, tipoExtracaoOcr} idêntica                          3   
                                                   {result}                                   campos a mais                     1   
nº10 · U_campo_inexistente validar_quebra_sigilo   {justificativa, vazamento_sigilo}          idêntica                          7   

                                                                                                                            2025-12  \
unidade                    função                  nível (chaves de topo)                     comparação com a forma comum            
nº2 · U_contrato_dict      get_available_documents {result}                                   idêntica                            8   
                                                   {hashDocumento, metadado, tipoExtracaoOcr} idêntica                            0   
                                                   {result}                                   campos a mais                       0   
nº10 · U_campo_inexistente validar_quebra_sigilo   {justificativa, vazamento_sigilo}          idêntica                            0   

                                                                                                                            2026-04  \
unidade                    função                  nível (chaves de topo)                     comparação com a forma comum            
nº2 · U_contrato_dict      get_available_documents {result}                                   idêntica                            7   
                                                   {hashDocumento, metadado, tipoExtracaoOcr} idêntica                            0   
                                                   {result}                                   campos a mais                       0   
nº10 · U_campo_inexistente validar_quebra_sigilo   {justificativa, vazamento_sigilo}          idêntica                            1   

                                                                                                                            2026-05  \
unidade                    função                  nível (chaves de topo)                     comparação com a forma comum            
nº2 · U_contrato_dict      get_available_documents {result}                                   idêntica                           23   
                                                   {hashDocumento, metadado, tipoExtracaoOcr} idêntica                            0   
                                                   {result}                                   campos a mais                       0   
nº10 · U_campo_inexistente validar_quebra_sigilo   {justificativa, vazamento_sigilo}          idêntica                            2   

                                                                                                                            2026-06  \
unidade                    função                  nível (chaves de topo)                     comparação com a forma comum            
nº2 · U_contrato_dict      get_available_documents {result}                                   idêntica                           37   
                                                   {hashDocumento, metadado, tipoExtracaoOcr} idêntica                            3   
                                                   {result}                                   campos a mais                       0   
nº10 · U_campo_inexistente validar_quebra_sigilo   {justificativa, vazamento_sigilo}          idêntica                            4   

                                                                                 

erros  \
unidade                    função                  nível (chaves de topo)                              
nº2 · U_contrato_dict      get_available_documents {result}                                       86   
                                                   {hashDocumento, metadado, tipoExtracaoOcr}      3   
nº10 · U_campo_inexistente validar_quebra_sigilo   {justificativa, vazamento_sigilo}               7   

                                                                                                                                   meses observados  \
unidade                    função                  nível (chaves de topo)                                                                             
nº2 · U_contrato_dict      get_available_documents {result}                                    2025-12, 2026-04, 2026-05, 2026-06, 2026-07, 2026-08   
                                                   {hashDocumento, metadado, tipoExtracaoOcr}                                               2026-06   
nº10 · U_campo_inexistente validar_quebra_sigilo   {justificativa, vazamento_sigilo}                                      2026-04, 2026-05, 2026-06   

                                                                                               nº de meses  \
unidade                    função                  nível (chaves de topo)                                    
nº2 · U_contrato_dict      get_available_documents {result}                                              6   
                                                   {hashDocumento, metadado, tipoExtracaoOcr}            1   
nº10 · U_campo_inexistente validar_quebra_sigilo   {justificativa, vazamento_sigilo}                     3   

                                                                                               formas distintas  \
unidade                    função                  nível (chaves de topo)                                         
nº2 · U_contrato_dict      get_available_documents {result}                                                   2   
                                                   {hashDocumento, metadado, tipoExtracaoOcr}                 1   
nº10 · U_campo_inexistente validar_quebra_sigilo   {justificativa, vazamento_sigilo}                          1   

                                                                                               idênticas  \
unidade                    função                  nível (chaves de topo)                                  
nº2 · U_contrato_dict      get_available_documents {result}                                           85   
                                                   {hashDocumento, metadado, tipoExtracaoOcr}          3   
nº10 · U_campo_inexistente validar_quebra_sigilo   {justificativa, vazamento_sigilo}                   7   

                                                                                               campos a mais  \
unidade                    função                  nível (chaves de topo)                                      
nº2 · U_contrato_dict      get_available_documents {result}                                                1   
                                                   {hashDocumento, metadado, tipoExtracaoOcr}              0   
nº10 · U_campo_inexistente validar_quebra_sigilo   {justificativa, vazamento_sigilo}                       0   

                                                                                               contradições  \
unidade                    função                  nível (chaves de topo)                                     
nº2 · U_contrato_dict      get_available_documents {result}                                               0   
                                                   {hashDocumento, metadado, tipoExtracaoOcr}             0   
nº10 · U_campo_inexistente validar_quebra_sigilo   {justificativa, vazamento_sigilo}                      0   

                                 

evidência: 11 casos e 2 tabela(s) em resultados/evidencia/11.5_estabilidade/ — crus e visões por caso: `python drill_down.py evidencia 11.5_estabilidade`


### 11.6 · Passo 5 — status: derivado e checado, ou parcial?

**O que faz.** Aplica a régua pré-registrada a cada sub-unidade candidata: **`derived-and-checked`** se ≥90% das
ocorrências tiveram o objeto lido no Passo 2 **ou** confirmado por evidência indireta **e** o Passo 4 não achou
contradição; senão **`parcial`**, com a cobertura exata. Uma ocorrência conta como lida/confirmada se pelo menos um
dos seus erros contar. **Confirmação indireta (17/09):** quando a mensagem é `AttributeError` de "dict iterado como
lista" (`Object <chave> has no attribute get`), ela não imprime o objeto inteiro, só a chave em que a iteração
parou — se essa chave já é conhecida do schema (derivado de OUTROS erros da mesma ferramenta), conta como
confirmação; regra geral, testada contra qualquer chave, não escrita pra estes casos específicos.

**Como ler.** A primeira tabela dá cobertura (adotada e estrita — só objeto lido de fato, sem a confirmação indireta),
contradições, campos a mais e o status nas duas leituras. A segunda lista o resíduo de verdade: os erros sem nenhuma
evidência (nem objeto lido, nem chave confirmada).


In [8]:
# 11.6 · Passo 5 — status: derivado e checado, ou parcial?
STATUS, RESIDUO = status_passo5(SCH, EST, CANDIDATAS, CHAVES_REAIS)
display(STATUS)
display(RESIDUO.assign(unidade=RESIDUO["unidade"].map(rotulo))[["unidade", "funcao_origem", "mes", "idx", "pedido", "motivo_residuo"]]
        .rename(columns={"funcao_origem": "função", "mes": "mês", "motivo_residuo": "por que não foi lido"}))

# evidência (ver 11.0): todos os erros do resíduo
registrar_evidencia(
    "11.6_status", "§11.6",
    "a cobertura de leitura de cada candidata e o motivo de cada erro do resíduo (objeto não lido).",
    "todos os erros do resíduo das sub-unidades candidatas.",
    RESIDUO.assign(ferramenta=lambda d: d["funcao_origem"], motivo=lambda d: "resíduo: " + d["motivo_residuo"])
      [["exec_id", "role", "idx", "mes", "unidade", "ferramenta", "pedido", "motivo"]],
    {"status": STATUS.reset_index(), "residuo": RESIDUO})


,,erros,erros lidos ou confirmados,confirmados por AttributeError,ocorrências,ocorrências lidas/confirmadas,cobertura,cobertura estrita,contradições (Passo 4),campos a mais (Passo 4),status,status na leitura estrita
unidade,função,,,,,,,,,,,
nº2 · U_contrato_dict,get_available_documents,91,91,2,86,86,100.0%,100.0%,0,1,derived-and-checked,parcial
nº10 · U_campo_inexistente,validar_quebra_sigilo,7,7,0,7,7,100.0%,100.0%,0,0,derived-and-checked,derived-and-checked


,unidade,função,mês,idx,pedido,por que não foi lido


evidência: 0 casos e 2 tabela(s) em resultados/evidencia/11.6_status/ — crus e visões por caso: `python drill_down.py evidencia 11.6_status`


### 11.7 · Passo 6 — amostra para conferir o schema contra o trace cru

**O que faz.** Sorteia, com semente fixa, 5 erros por sub-unidade candidata — 1 deles do resíduo do Passo 5, quando há —
e grava a pasta de evidência com o trace cru de cada um. Para cada caso, confere três fontes: **(1)** o objeto que a
mensagem de erro imprime tem a forma comum do Passo 4 (é o mesmo leitor do Passo 2: mede consistência, não acerto);
**(2)** as chaves do schema aparecem no **log de `print`** que o agente recebeu, um texto diferente da mensagem de erro;
**(3)** o conserto do Passo 3 lê uma chave do schema.

**Como ler.** Uma linha por caso sorteado. A verificação humana pré-registrada — abrir o cru e conferir — continua
pendente; esta tabela organiza o que olhar, e a pasta `resultados/evidencia/11.7_amostra_passo6/` tem o cru.


In [9]:
# 11.7 · Passo 6 — amostra para conferir o schema contra o trace cru
AMOSTRA_P6 = amostra_passo6(SCH, CANDIDATAS)
CONF_P6 = conferir_amostra(AMOSTRA_P6, EST, AUTOCORR, STEPS_P1)
display(CONF_P6.drop(columns=["exec_id", "role"]))

# evidência (ver 11.0): os casos sorteados
registrar_evidencia(
    "11.7_amostra_passo6", "§11.7",
    "o schema derivado (Passos 2 e 4) bate com o trace cru numa amostra sorteada de cada sub-unidade candidata.",
    f"5 erros por sub-unidade candidata, sorteados com `random.Random({SEMENTE_P6})`: 1 do resíduo do Passo 5, quando há, "
    "e o resto entre os erros com objeto lido.",
    AMOSTRA_P6.assign(ferramenta=lambda d: d["funcao_origem"],
                      chaves=[";".join(sorted(chaves_da_forma(chaves_do_schema(EST, u, fn)[1])))
                              for u, fn in zip(AMOSTRA_P6["unidade"], AMOSTRA_P6["funcao_origem"])])
      [["exec_id", "role", "idx", "mes", "unidade", "ferramenta", "pedido", "forma", "chaves", "motivo"]],
    {"conferencia_amostra": CONF_P6},
    nota="**Verificação humana (Passo 6 pré-registrado — pendente).** Para cada caso: abrir a visão "
         "`derivados/<exec>_<role>_idx<n>.json`, seguir o caminho da mensagem de erro e do log de print até "
         "`crus/<exec_id>.json`, e conferir no cru que o objeto tem as chaves da coluna `chaves` de `casos.csv`. "
         "Anotar o resultado em `03-procedimento-validacao.md` §1.9.")


,unidade,função,mês,exec,idx,motivo,(1) objeto da mensagem tem a forma comum,(2) chaves do schema no log de print,(3) conserto lê chave do schema
0,nº2 · U_contrato_dict,get_available_documents,2026-05,cbc5d62d…,1,sorteado entre os erros com objeto lido,sim,sim (6/6),sim — lê result
1,nº2 · U_contrato_dict,get_available_documents,2026-06,05a8fe50…,1,sorteado entre os erros com objeto lido,sim,sim (6/6),sim — lê result
2,nº2 · U_contrato_dict,get_available_documents,2026-06,3f44a68b…,2,resíduo do Passo 5,— (objeto não lido: resíduo),sim (6/6),— (não aplicável)
3,nº2 · U_contrato_dict,get_available_documents,2026-06,c8627731…,1,sorteado entre os erros com objeto lido,sim,sim (6/6),sim — lê result
4,nº2 · U_contrato_dict,get_available_documents,2026-07,99ab53e8…,1,sorteado entre os erros com objeto lido,sim,sim (6/6),sim — lê result
5,nº10 · U_campo_inexistente,validar_quebra_sigilo,2026-04,d3d64194…,1,sorteado entre os erros com objeto lido,sim,sim (2/2),sim — lê vazamento_sigilo
6,nº10 · U_campo_inexistente,validar_quebra_sigilo,2026-05,1be966e7…,6,sorteado entre os erros com objeto lido,sim,sim (2/2),sim — lê vazamento_sigilo
7,nº10 · U_campo_inexistente,validar_quebra_sigilo,2026-05,8dd410c8…,1,sorteado entre os erros com objeto lido,sim,sim (2/2),sim — lê vazamento_sigilo
8,nº10 · U_campo_inexistente,validar_quebra_sigilo,2026-06,21a4fa6c…,2,sorteado entre os erros com objeto lido,sim,sim (2/2),.get com padrão <var>.get('vazamento_sigilo')
9,nº10 · U_campo_inexistente,validar_quebra_sigilo,2026-06,9be7b413…,5,sorteado entre os erros com objeto lido,sim,sim (2/2),sim — lê vazamento_sigilo


evidência: 10 casos e 1 tabela(s) em resultados/evidencia/11.7_amostra_passo6/ — crus e visões por caso: `python drill_down.py evidencia 11.7_amostra_passo6`


### 11.8 · Passos 7 e 8 — o registro final de cada candidata

**O que faz.** Monta, para cada sub-unidade candidata, o registro no schema de `01-racionais.md` §8, só com o que os
passos anteriores derivaram. **Passo 7:** `description` (o que aconteceu) e `correction_guidance` (o que fazer) saem de
um molde fixo preenchido com a forma comum (Passo 4), o pedido mais comum do agente (Passo 2), a chave para a qual ele
trocou (Passo 3) e o que o bloco da ferramenta no prompt declara (11.3). **Passo 8:** `location` (até 10 casos,
alternando os meses), `evidence` (os 3 primeiros, de meses diferentes, só estrutura), `occurrences`, `status` (Passo 5),
`validation` (o resultado de cada passo) e `impact` = `null` (decisão registrada no §9).

**Como ler.** Um registro por candidata. O registro não decide o destino da nº10: quando a correção contradiz o system
prompt, `validation.destino` diz isso e deixa a decisão (memória × harness) em aberto. Gravado em
`resultados/unidades_memoria.json`; na tela, o `exec_id` aparece encurtado.


In [10]:
# 11.8 · Passos 7 e 8 — o registro final de cada candidata
REGISTROS = [registro_final(u, fn, ATRIB, EU, SCH, AUTOCORR, PRESENCA, EST, STATUS, CONF_P6, STEPS_P1) for u, fn in CANDIDATAS]
with open("resultados/unidades_memoria.json", "w", encoding="utf-8") as f:
    json.dump(REGISTROS, f, ensure_ascii=False, indent=2)
print(json.dumps(encurtar_ids(REGISTROS), ensure_ascii=False, indent=2))

# evidência (ver 11.0): os casos citados em `evidence` de cada registro
registrar_evidencia(
    "11.8_registro_final", "§11.8",
    "os casos citados no campo `evidence` do registro final de cada candidata.",
    "os 3 primeiros casos de `location` de cada registro — o primeiro de cada um dos 3 primeiros meses com erro.",
    pd.DataFrame([{**{k: ev[k] for k in ("exec_id", "role", "idx", "mes")}, "unidade": u, "ferramenta": fn,
                   "pedido": ev["pedido"], "forma": ev["forma_do_objeto_indexado"], "motivo": "citado em `evidence` do registro final"}
                  for (u, fn), r in zip(CANDIDATAS, REGISTROS) for ev in r["evidence"]]),
    {})
with open(os.path.join(EVIDENCIA, "11.8_registro_final", "derivados", "unidades_memoria.json"), "w", encoding="utf-8") as f:
    json.dump(REGISTROS, f, ensure_ascii=False, indent=2)


[
  {
    "category": "nº2 · Retorno das ferramentas de documento é dict (factual · ambiente)",
    "scope": {
      "role": "ConversationAgent",
      "tool": "get_available_documents"
    },
    "location": {
      "total_erros": 91,
      "casos": [
        {
          "exec_id": "1eed2931…",
          "role": "ConversationAgent",
          "idx": 1,
          "mes": "2025-12"
        },
        {
          "exec_id": "0cd5e2c0…",
          "role": "ConversationAgent",
          "idx": 1,
          "mes": "2026-04"
        },
        {
          "exec_id": "0572af0b…",
          "role": "ConversationAgent",
          "idx": 1,
          "mes": "2026-05"
        },
        {
          "exec_id": "008d142f…",
          "role": "ConversationAgent",
          "idx": 1,
          "mes": "2026-06"
        },
        {
          "exec_id": "124ca0ef…",
          "role": "ConversationAgent",
          "idx": 1,
          "mes": "2026-07"
        },
        {
          "exec_id": "575122c5…"

### 11.9 · Análise funda — o que o payload entregue diz sobre `validar_quebra_sigilo`

**O que faz.** Pré-registro completo em `06-racionais-mineracao-unidades-n2-n10.md` §9 ("Análise funda da nº10"). Os Passos 1–8 acima só
enxergam erro que levanta exceção — os 7 já contados. Esta análise mede a outra metade: leituras do retorno que
**não** quebram, e o que a esteira **entregou de verdade** nessas execuções.

Duas medidas, nessa ordem de confiança: **(1) leitura de código** — para cada `x = validar_quebra_sigilo(...)`,
classifica a leitura de `x` numa das quatro classes vivas ((a) chave real, (b) com erro, (c) sem erro — o alvo —,
(d) objeto inteiro repassado), tratando `.get(a, .get(b))` como uma **cadeia** que resolve na primeira chave real,
não como duas leituras independentes — foi corrigindo esse ponto que um achado errado da primeira rodada (fora do
notebook) caiu; a retratação está em `03-procedimento-validacao.md` §1.10. **(2) o payload entregue** —
`action_output['request']['quebra_sigilo']` do último step final de cada execução: não é inferência, é o que saiu.
Onde as duas medidas existem, a (2) decide; a (1) só explica o mecanismo.

**Como ler.** A primeira tabela é a leitura de código, por classe — mostra a *forma* do risco. A segunda é o payload
entregue, por categoria de valor — mostra o *desfecho*. A terceira soma execuções cujo valor entregue não é
`SIM`/`NAO`: é o número que decide `destino`/`impact` da nº10, pela regra fixada no pré-registro (≥1 caso →
harness). A quarta é a grafia (`"NÃO"` × `'NAO'`), com a linha em que cada step falhou, pra mostrar se a comparação
chegou a rodar. Por fim, a célula regrava `resultados/unidades_memoria.json` com o desfecho — o registro da nº10 sai desta seção já com `destino`/`impact` preenchidos, não editado à mão.





In [11]:
# 11.9 · Análise funda — o que o payload entregue diz sobre validar_quebra_sigilo
FN_QS, CAMPO_QS = "validar_quebra_sigilo", "quebra_sigilo"
DECL_QS = re.compile(r"def\s+" + FN_QS + r"\s*\(")
CHAVES_REAIS_QS = CHAVES_REAIS.get(FN_QS, set())

PAPEIS_QS = sorted({(r["exec_id"], r["role"]) for r in RAW if DECL_QS.search(r["sysprompt"] or "")})
CHAMOU_QS = sorted(p for p in PAPEIS_QS if any(chamadas_de(STEPS_P1[p][idx]["code"], FN_QS) for idx in STEPS_P1[p]))
NUNCA_CHAMOU_QS = [p for p in PAPEIS_QS if p not in CHAMOU_QS]
print(f"papéis que declaram {FN_QS}: {len(PAPEIS_QS)} · chamaram: {len(CHAMOU_QS)} · "
      f"declararam e nunca chamaram: {len(NUNCA_CHAMOU_QS)}")

linhas = []
for ex, role in CHAMOU_QS:
    idx_ultimo_final = max((idx for idx in STEPS_P1[(ex, role)] if STEPS_P1[(ex, role)][idx]["is_final"]), default=None)
    idx_ref = idx_ultimo_final if idx_ultimo_final is not None else max(STEPS_P1[(ex, role)])
    mes = STEPS_P1[(ex, role)][idx_ref]["mes"]
    cls = classificar_campo_final(STEPS_P1, ex, role, CAMPO_QS, FN_QS, CHAVES_REAIS_QS)
    valor = payload_entregue_qs(STEPS_P1, ex, role, CAMPO_QS)
    linhas.append({"exec_id": ex, "role": role, "idx": idx_ref, "mes": mes,
                   "classe_leitura": cls[0] if cls else "(campo final não construído neste papel)",
                   "chave_lida": chave_segura(cls[1]) if cls and cls[1] else None,
                   "motivo_leitura": cls[2] if cls else "", "categoria_entregue": categoria_valor_entregue(valor)})
QS = pd.DataFrame(linhas)

print("\nLeitura de código — como o campo final é construído (explicativo; a decisão é pelo payload abaixo):")
display(QS.groupby("classe_leitura").agg(execucoes=("classe_leitura", "size"), meses=("mes", "nunique"))
        .sort_values("execucoes", ascending=False))

print("\nPayload entregue — o que a esteira recebeu de verdade, medido em action_output.request:")
display(QS.groupby("categoria_entregue").agg(execucoes=("categoria_entregue", "size"), meses=("mes", "nunique"))
        .sort_values("execucoes", ascending=False))

ENTREGUES_QS = QS[QS["categoria_entregue"] != "ausente (None)"]
NAO_CONFORME_QS = ENTREGUES_QS[~ENTREGUES_QS["categoria_entregue"].str.contains(r"\(valor esperado\)")]
N_NC, N_TOT = len(NAO_CONFORME_QS), len(ENTREGUES_QS)
print(f"\n>>> {N_NC}/{N_TOT} respostas entregues ({N_NC/N_TOT:.0%}) com o campo '{CAMPO_QS}' fora do esperado, "
      f"em {NAO_CONFORME_QS['mes'].nunique()} meses — nenhuma com erro registrado.")
display(NAO_CONFORME_QS[["exec_id", "role", "mes", "categoria_entregue"]]
        .assign(exec_id=lambda d: d["exec_id"].str[:8] + "…"))

print("\nGrafia do valor — comparações com literal que só bate com o real depois de normalizar acento/caixa:")
grafia_rows = [g for ex, role in CHAMOU_QS for g in comparacoes_grafia_qs(STEPS_P1, ex, role, FN_QS, {"NAO", "SIM"})]
GRAFIA_QS = pd.DataFrame(grafia_rows, columns=["exec_id", "role", "idx", "mes", "literal", "alcance"])
if len(GRAFIA_QS):
    display(GRAFIA_QS.assign(exec_id=lambda d: d["exec_id"].str[:8] + "…")
            [["exec_id", "role", "idx", "mes", "literal", "alcance"]])
else:
    print("nenhuma comparação de grafia divergente encontrada")

# a decisão — regra fixada no pré-registro, 06-racionais-mineracao-unidades-n2-n10.md §9: ≥1 caso não conforme → harness
if N_NC >= 1:
    DESTINO_QS = "harness"
    IMPACT_QS = (f"{N_NC}/{N_TOT} respostas entregues ({N_NC/N_TOT:.0%}) com o campo regulatório de "
                f"{CAMPO_QS} inválido, sem nenhum erro registrado, em {NAO_CONFORME_QS['mes'].nunique()} meses.")
else:
    DESTINO_QS = "memória — nenhuma resposta entregue com o campo fora do esperado nesta análise"
    IMPACT_QS = None
NOTA_QS = (f"§11.9 (17/09): payload entregue medido em action_output.request — {N_NC}/{N_TOT} respostas fora do "
          f"esperado. Regra pré-registrada (≥1 caso → harness), 06-racionais-mineracao-unidades-n2-n10.md §9.")
DESFECHO_11_9 = {("U_campo_inexistente", FN_QS): {"destino": DESTINO_QS, "impact": IMPACT_QS, "nota": NOTA_QS}}
print(f"\n>>> destino: {DESTINO_QS}")
print(f">>> impact: {IMPACT_QS}")

# evidência (ver 11.0): todas as não conformes + grafia, mais o primeiro caso de cada classe de leitura
casos_qs = pd.concat([
    NAO_CONFORME_QS.assign(motivo="payload não conforme: " + NAO_CONFORME_QS["categoria_entregue"]),
    GRAFIA_QS.assign(motivo="grafia divergente: " + GRAFIA_QS["literal"] + " (" + GRAFIA_QS["alcance"] + ")"),
    primeiro_por(QS, ["classe_leitura"]).pipe(lambda d: d.assign(motivo="primeiro caso desta classe de leitura: " + d["classe_leitura"])),
], ignore_index=True)
registrar_evidencia(
    "11.9_leituras_quebra_sigilo", "§11.9",
    "quantas execuções que declaram `validar_quebra_sigilo` entregam, no payload final, um campo `quebra_sigilo` "
    "que não é `SIM`/`NAO` — sem nenhum erro registrado — e a comparação de grafia do valor.",
    "todas as execuções cujo payload entregue não é `SIM`/`NAO`, todas as comparações de grafia divergente, "
    "mais o primeiro caso de cada classe de leitura de código, para comparação.",
    casos_qs.assign(ferramenta=FN_QS)[["exec_id", "role", "idx", "mes", "ferramenta", "motivo"]],
    {"quebra_sigilo_por_papel": QS, "grafia": GRAFIA_QS})

# o desfecho desta análise decide destino/impact da nº10 (06-racionais-mineracao-unidades-n2-n10.md §9) — refaz o registro final da §11.8
# com essa informação agora disponível, e regrava unidades_memoria.json (mesmo arquivo, valor atualizado).
REGISTROS = [registro_final(u, fn, ATRIB, EU, SCH, AUTOCORR, PRESENCA, EST, STATUS, CONF_P6, STEPS_P1,
                            desfecho=DESFECHO_11_9) for u, fn in CANDIDATAS]
with open("resultados/unidades_memoria.json", "w", encoding="utf-8") as f:
    json.dump(REGISTROS, f, ensure_ascii=False, indent=2)
print("\nunidades_memoria.json regravado com o desfecho da §11.9:")
print(json.dumps(encurtar_ids(REGISTROS), ensure_ascii=False, indent=2))





papéis que declaram validar_quebra_sigilo: 26 · chamaram: 21 · declararam e nunca chamaram: 5



Leitura de código — como o campo final é construído (explicativo; a decisão é pelo payload abaixo):


,execucoes,meses
classe_leitura,,
(a) chave real,11,4
(d) objeto inteiro repassado,7,2
(e) literal fixo,2,1
"(c) chave declarada, sem erro",1,1



Payload entregue — o que a esteira recebeu de verdade, medido em action_output.request:


,execucoes,meses
categoria_entregue,,
'NAO' (valor esperado),11,4
"objeto inteiro (justificativa, vazamento_sigilo)",7,2
'SIM' (valor esperado),1,1
"<texto livre, 203 chars>",1,1
string vazia,1,1



>>> 9/21 respostas entregues (43%) com o campo 'quebra_sigilo' fora do esperado, em 3 meses — nenhuma com erro registrado.


,exec_id,role,mes,categoria_entregue
0,0578177f…,RespostaBacen,2026-06,"objeto inteiro (justificativa, vazamento_sigilo)"
3,2bb6ea3a…,RespostaBacen,2025-12,"<texto livre, 203 chars>"
6,6687b90b…,RespostaBacen,2026-06,"objeto inteiro (justificativa, vazamento_sigilo)"
9,827713a4…,RespostaBacen,2026-06,"objeto inteiro (justificativa, vazamento_sigilo)"
11,95344639…,RespostaBacen,2026-06,"objeto inteiro (justificativa, vazamento_sigilo)"
13,a6a62b99…,RespostaBacen,2026-06,"objeto inteiro (justificativa, vazamento_sigilo)"
15,c42e6076…,RespostaBacen,2026-05,"objeto inteiro (justificativa, vazamento_sigilo)"
17,dbc472b0…,RespostaBacen,2025-12,string vazia
20,f69cd0ee…,RespostaBacen,2026-06,"objeto inteiro (justificativa, vazamento_sigilo)"



Grafia do valor — comparações com literal que só bate com o real depois de normalizar acento/caixa:


,exec_id,role,idx,mes,literal,alcance
0,21a4fa6c…,RespostaBacen,2,2026-06,NÃO,indeterminada — mesma linha da falha
1,21a4fa6c…,RespostaBacen,3,2026-06,NÃO,avaliada
2,21a4fa6c…,RespostaBacen,3,2026-06,Não,avaliada
3,21a4fa6c…,RespostaBacen,3,2026-06,Nao,avaliada
4,50d6c3bd…,RespostaBacen,2,2026-06,NÃO,indeterminada — mesma linha da falha



>>> destino: harness
>>> impact: 9/21 respostas entregues (43%) com o campo regulatório de quebra_sigilo inválido, sem nenhum erro registrado, em 3 meses.
evidência: 13 casos e 2 tabela(s) em resultados/evidencia/11.9_leituras_quebra_sigilo/ — crus e visões por caso: `python drill_down.py evidencia 11.9_leituras_quebra_sigilo`

unidades_memoria.json regravado com o desfecho da §11.9:
[
  {
    "category": "nº2 · Retorno das ferramentas de documento é dict (factual · ambiente)",
    "scope": {
      "role": "ConversationAgent",
      "tool": "get_available_documents"
    },
    "location": {
      "total_erros": 91,
      "casos": [
        {
          "exec_id": "1eed2931…",
          "role": "ConversationAgent",
          "idx": 1,
          "mes": "2025-12"
        },
        {
          "exec_id": "0cd5e2c0…",
          "role": "ConversationAgent",
          "idx": 1,
          "mes": "2026-04"
        },
        {
          "exec_id": "0572af0b…",
          "role": "ConversationAg

### 11.10 · Análise funda — leituras silenciosas do retorno de `get_available_documents` (nº2)

**O que faz.** Pré-registro em `06-racionais-mineracao-unidades-n2-n10.md` §9 ("Análise funda da nº2"). Estende o Passo 1 da §11 (que só olhava os 91 erros já conhecidos) para o **universo completo**: toda chamada a `get_available_documents`, rastreando a variável entre steps do mesmo papel (mesma regra de `acessos()`), separando quem lê a chave certa (`result`) de quem lê uma chave errada — com ou sem guarda de tipo, com ou sem `try/except`, com ou sem erro no step. Cruza as leituras erradas e silenciosas com o único detector de resposta degenerada já validado no pipeline (`DEGENERADO`), aplicado à resposta **real** do `managerAgent` (não à saída intermediária do `ConversationAgent`).

**Como ler.** A tabela por `(real, guardado, protegido)` separa: leitura correta · errada mas dentro de uma guarda de tipo que sempre escolhe o ramo certo (os 17 casos já documentados, dead code no ramo errado) · errada mas protegida por `try/except` · errada e sem proteção nenhuma — desta última, quem já está entre os 91 erros conhecidos (visível) e quem não está (silenciosa: rodou, devolveu algo errado, nenhuma exceção). **Diferente da §11.9, esta análise não decide `destino`/`impact`** — só documenta `analise_funda`, porque o cruzamento com `DEGENERADO` não confirma dano à resposta entregue (ver célula abaixo e `03-procedimento-validacao.md`).




In [12]:
# 11.10 · Análise funda — leituras silenciosas do retorno de get_available_documents (no2)
# Pre-registro: 06-racionais-mineracao-unidades-n2-n10.md §9, "Análise funda da nº2". Estende o Passo 1 da §11 (que só olhava os 91 erros
# já conhecidos) para o universo completo de chamadas, e cruza as leituras erradas e silenciosas com o único
# detector de resposta degenerada já validado no pipeline (DEGENERADO), na resposta real do managerAgent.
FN2 = "get_available_documents"
CHAVE_TOPO_REAL_2 = "result"
DECL_2 = re.compile(r"def\s+" + FN2 + r"\s*\(")

VISIVEIS_CONHECIDOS_2 = {(r["exec_id"], r["role"], r["idx"])
                        for r in ATRIB[(ATRIB["unidade"] == "U_contrato_dict") & (ATRIB["funcao_origem"] == FN2)].to_dict("records")}
print(f"erros já conhecidos (nº2, {FN2}): {len(VISIVEIS_CONHECIDOS_2)} — usado só para conferência, não redefinido aqui.")

PAPEIS_2 = sorted({(r["exec_id"], r["role"]) for r in RAW if DECL_2.search(r["sysprompt"] or "")})


def raiz_nome_2(v):
    while isinstance(v, (ast.Subscript, ast.Attribute)):
        v = v.value
    return v.id if isinstance(v, ast.Name) else None


def guardas_chave_fantasma(arv, var):
    """[(chave_testada, ramo_sempre_executa_ignora_var)] para If/IfExp cujo teste é `'chave' in <var>` (ou
    `not in`) e 'chave' não está no schema real (só `result` existe no envelope) — o ramo que dependeria da
    chave real nunca roda porque a condição é sempre falsa (schema estável, §9 Passo 4)."""
    out = []
    for n in ast.walk(arv):
        if not isinstance(n, (ast.If, ast.IfExp)):
            continue
        teste = n.test
        if not (isinstance(teste, ast.Compare) and len(teste.ops) == 1 and isinstance(teste.ops[0], (ast.In, ast.NotIn))):
            continue
        esquerda, direita = teste.left, teste.comparators[0]
        if not (isinstance(esquerda, ast.Constant) and isinstance(esquerda.value, str)):
            continue
        if raiz_nome_2(direita) != var:
            continue
        chave = esquerda.value
        if chave == CHAVE_TOPO_REAL_2:
            continue  # chave real — guarda normal (as 17 já conhecidas), não fantasma
        vivo = n.orelse if isinstance(teste.ops[0], ast.In) else n.body
        if not vivo:
            ignora = True  # If sem else — nada acontece quando a condição (sempre falsa) não bate
        else:
            blocos = vivo if isinstance(vivo, list) else [vivo]
            nomes_no_vivo = {x.id for b in blocos for x in ast.walk(b) if isinstance(x, ast.Name)}
            ignora = var not in nomes_no_vivo
        out.append({"chave_testada": chave, "ramo_vivo_ignora_var": ignora})
    return out


linhas2, linhas_g = [], []
for (ex, role) in PAPEIS_2:
    ps = STEPS_P1[(ex, role)]
    idxs_ordenados = sorted(ps)
    for idx0 in idxs_ordenados:
        chamadas = chamadas_de(ps[idx0]["code"], FN2)
        if not chamadas:
            continue
        for var, _linha in chamadas:
            for idx in idxs_ordenados:
                if idx < idx0:
                    continue
                st = ps[idx]
                try:
                    arv = ast.parse(st["code"])
                except SyntaxError:
                    continue
                for forma, chave, protegido, padrao, guardado in acessos(arv, var, FN2):
                    chave_str = str(chave)
                    linhas2.append({"exec_id": ex, "role": role, "idx_chamada": idx0, "idx_leitura": idx,
                                    "mes": st["mes"], "forma": forma, "chave": chave_str,
                                    "real": chave_str == CHAVE_TOPO_REAL_2, "protegido": protegido, "guardado": guardado,
                                    "tem_erro_no_step": bool(st["err_msg"]),
                                    "visivel_conhecido": (ex, role, idx) in VISIVEIS_CONHECIDOS_2})
                for g in guardas_chave_fantasma(arv, var):
                    linhas_g.append({"exec_id": ex, "role": role, "idx_chamada": idx0, "idx_leitura": idx,
                                     "mes": st["mes"], "chave_testada": g["chave_testada"],
                                     "descarta_documentos": g["ramo_vivo_ignora_var"]})
                # reatribuição que corta o rastreio entre steps — mesmo critério de acessos() dentro de um step
                cortou = False
                for n in ast.walk(arv):
                    if isinstance(n, (ast.Assign, ast.AnnAssign)):
                        alvo = n.targets[0] if isinstance(n, ast.Assign) else n.target
                        if isinstance(alvo, ast.Name) and alvo.id == var and origem(n.value) != ("call", FN2):
                            cortou = True
                if cortou:
                    break

L2 = pd.DataFrame(linhas2)
print(f"\ntotal de leituras rastreadas (universo completo, todas as {len(PAPEIS_2)} execuções que declaram {FN2}): {len(L2)}")
display(L2.groupby(["real", "guardado", "protegido"], dropna=False).size().to_frame("leituras"))

errada2 = L2[~L2["real"]]
sem_protecao_2 = errada2[(~errada2["guardado"]) & (~errada2["protegido"])]
print("\nchave/índice errado, sem guarda de tipo e sem try/except — visível (já conta nos 91) vs. silencioso (novo):")
display(sem_protecao_2.groupby(["visivel_conhecido", "tem_erro_no_step"], dropna=False).size().to_frame("leituras"))

silenciosas_2 = (sem_protecao_2[(~sem_protecao_2["visivel_conhecido"]) & (~sem_protecao_2["tem_erro_no_step"])]
                 [["exec_id", "role", "idx_chamada", "idx_leitura", "mes", "forma", "chave"]]
                 .drop_duplicates(subset=["exec_id", "role", "idx_leitura"]))
print(f"\n>>> {len(silenciosas_2)} ocorrências SILENCIOSAS confirmadas (leitura errada, sem guarda, sem try/except, "
      "sem exceção no step, fora dos 91 erros já conhecidos) — universo exaustivo, não amostra.")

# DEGENERADO só pega recusa EXPLÍCITA ("não encontrado na base"...). Um `.get(chave_errada)` sem plano B devolve
# None em silêncio — isso pode vazar pro final_answer como um "None"/"null" literal, ou como o dict/objeto inteiro
# stringificado (o mesmo mecanismo que a nº10 achou, classe "objeto inteiro repassado", §6.2), sem nenhuma frase
# de recusa. Checagem adicional, estrutural (nunca conteúdo de documento): presença literal de "None"/"null", e
# um regex frouxo pra "parece um dict/objeto Python impresso" (chave entre aspas seguida de `:` dentro de `{}`).
NULO_LEAK = re.compile(r"\bNone\b|\bnull\b", re.I)
DICT_LEAK = re.compile(r"\{[^{}]{0,300}:[^{}]{0,300}\}")

DEGENERADO_POR_CASO_2, NULO_POR_CASO_2, DICT_POR_CASO_2 = [], [], []
for _, row in silenciosas_2.iterrows():
    mgr_finais = [s for s in STEPS_P1.get((row["exec_id"], "managerAgent"), {}).values() if s.get("is_final")]
    txt_final_real = str(mgr_finais[-1].get("out") or "") if mgr_finais else ""
    DEGENERADO_POR_CASO_2.append(bool(DEGENERADO.search(txt_final_real)) if mgr_finais else None)
    NULO_POR_CASO_2.append(bool(NULO_LEAK.search(txt_final_real)) if mgr_finais else None)
    DICT_POR_CASO_2.append(bool(DICT_LEAK.search(txt_final_real)) if mgr_finais else None)
silenciosas_2 = silenciosas_2.assign(tem_final_manager=[d is not None for d in DEGENERADO_POR_CASO_2],
                                     degenerado_no_final_real=DEGENERADO_POR_CASO_2,
                                     none_ou_null_no_final_real=NULO_POR_CASO_2,
                                     parece_dict_no_final_real=DICT_POR_CASO_2)
display(silenciosas_2.assign(exec_id=lambda d: d["exec_id"].str[:8] + "…"))

n_confirmado_degenerado_2 = sum(bool(x) for x in DEGENERADO_POR_CASO_2)
n_nulo_2 = sum(bool(x) for x in NULO_POR_CASO_2)
n_dict_2 = sum(bool(x) for x in DICT_POR_CASO_2)
print(f"\n>>> Cruzando com DEGENERADO (único detector de resposta ruim já validado, sem lista de palavras nova): "
      f"{n_confirmado_degenerado_2}/{len(silenciosas_2)} têm a resposta REAL do managerAgent marcada como degenerada.")
print(f">>> Checagem adicional (None/null literal, ou dict/objeto aparentemente impresso) na MESMA resposta real: "
      f"{n_nulo_2}/{len(silenciosas_2)} com None/null · {n_dict_2}/{len(silenciosas_2)} parecendo dict impresso.")
print(">>> Pela régua já validada do pipeline, e por nenhuma das duas checagens adicionais, nenhuma das ocorrências")
print(">>> acima é hoje contável como 'resposta corrompida'. Isso RETRATA o rótulo 'confirmado: chega a final_answer")
print(">>> errado' da pasta exploratória 11.10 anterior — aquela leitura usava lista de palavras inventada sobre a")
print(">>> saída do ConversationAgent, não a resposta de verdade do managerAgent, nem um detector já validado.")

# segundo mecanismo, distinto do ".get sem plano B" acima: guarda cuja condição nunca é satisfeita pelo schema
# real — o ramo que usaria os documentos reais nunca roda, e eles são descartados por inteiro (não viram um
# valor errado). Quantificado exaustivamente aqui, não deixado como menção solta.
G = pd.DataFrame(linhas_g)
print(f"\nguardas sobre chave fantasma achadas (qualquer desfecho): {len(G)}")
descartes_2 = pd.DataFrame(columns=["exec_id", "role", "idx_chamada", "idx_leitura", "mes", "chave_testada"])
if len(G):
    display(G.groupby(["chave_testada", "descarta_documentos"], dropna=False).size().to_frame("ocorrências"))
    descartes_2 = (G[G["descarta_documentos"]][["exec_id", "role", "idx_chamada", "idx_leitura", "mes", "chave_testada"]]
                  .drop_duplicates(subset=["exec_id", "role", "idx_leitura"]))
print(f"\n>>> {len(descartes_2)} ocorrência(s), no trace inteiro, onde o ramo que SEMPRE roda ignora a variável por "
      "completo (documentos reais descartados, sem virar valor errado).")
DEG_DESCARTE_2 = []
for _, row in descartes_2.iterrows():
    mgr_finais = [s for s in STEPS_P1.get((row["exec_id"], "managerAgent"), {}).values() if s.get("is_final")]
    txt = str(mgr_finais[-1].get("out") or "") if mgr_finais else ""
    DEG_DESCARTE_2.append(bool(DEGENERADO.search(txt)) if mgr_finais else None)
    print(f"  {row['exec_id'][:8]}… chave='{row['chave_testada']}' mes={row['mes']} | "
          f"DEGENERADO no final real: {DEG_DESCARTE_2[-1]} | None/null: {bool(NULO_LEAK.search(txt))} | "
          f"parece dict: {bool(DICT_LEAK.search(txt))}")
n_deg_descarte_2 = sum(bool(x) for x in DEG_DESCARTE_2)
print(f">>> Abaixo do piso de recorrência que qualquer candidata a memória precisa (≥3 execuções e ≥2 meses, §7 "
      "Passo 5) — {} execução(ões), {} mês(es). Não vira unidade própria; fica documentado, não pendurado.".format(
          descartes_2["exec_id"].nunique(), descartes_2["mes"].nunique()))

NOTA_11_10 = (f"§11.10 (17/09): universo exaustivo (não amostra) — {len(silenciosas_2)} ocorrências de leitura "
             f"silenciosa tipo \".get sem plano B\" (chave errada, sem guarda, sem try/except, sem exceção) em "
             f"{silenciosas_2['exec_id'].nunique()} execuções, {silenciosas_2['mes'].nunique()} meses; "
             f"{n_confirmado_degenerado_2}/{len(silenciosas_2)} degeneradas por DEGENERADO, {n_nulo_2}/{len(silenciosas_2)} "
             f"com None/null literal, {n_dict_2}/{len(silenciosas_2)} parecendo dict impresso — três checagens "
             "independentes na resposta REAL do managerAgent, nenhuma achou corrupção; não confirma dano ao usuário "
             f"final; retrata o 'confirmado' da pasta exploratória anterior. Segundo mecanismo, também quantificado: "
             f"guarda sobre chave fantasma que descarta os documentos por inteiro — {len(descartes_2)} ocorrência(s) "
             f"no trace inteiro ({descartes_2['exec_id'].nunique()} execução(ões)), {n_deg_descarte_2}/{max(len(descartes_2),1)} "
             "degenerada(s) pelas mesmas três checagens; abaixo do piso de recorrência de qualquer candidata a "
             "memória (§7 Passo 5) — documentado, não vira unidade própria. Nenhuma das três checagens prova que a "
             "resposta está correta/completa (isso é groundedness, fora de escopo do v1) — só que não há recusa "
             "explícita nem corrupção estrutural óbvia no texto entregue.")
DESFECHO_11_10 = {("U_contrato_dict", FN2): {"nota": NOTA_11_10}}
print(f"\n>>> analise_funda (nº2): {NOTA_11_10}")

# evidência (ver 11.0): as ocorrências silenciosas confirmadas (os dois mecanismos) + o primeiro caso de cada
# combinação (real, guardado, protegido) do universo completo, para comparação.
casos_11_10 = pd.concat([
    silenciosas_2.rename(columns={"idx_leitura": "idx"})
        .assign(motivo="leitura silenciosa confirmada (chave errada, sem proteção, sem exceção)"),
    descartes_2.rename(columns={"idx_leitura": "idx"})
        .assign(motivo=lambda d: "guarda sobre chave fantasma '" + d["chave_testada"] + "' descarta os documentos"),
    primeiro_por(L2.rename(columns={"idx_leitura": "idx"}), ["real", "guardado", "protegido"])
        .assign(motivo=lambda d: "primeiro caso de (real=" + d["real"].astype(str) + ", guardado="
                + d["guardado"].astype(str) + ", protegido=" + d["protegido"].astype(str) + ")"),
], ignore_index=True)[["exec_id", "role", "idx", "mes", "motivo"]]
registrar_evidencia(
    "11.10_leituras_get_available_documents", "§11.10",
    "quantas leituras do retorno de `get_available_documents`, no universo completo de chamadas (não só os 91 "
    "erros já conhecidos), pedem uma chave errada sem guarda de tipo, sem try/except e sem levantar exceção — e "
    "se a resposta real do managerAgent bate o detector `DEGENERADO` já validado. Inclui também as guardas sobre "
    "chave fantasma que descartam os documentos por inteiro.",
    "as ocorrências silenciosas confirmadas dos dois mecanismos, mais o primeiro caso de cada combinação (real, "
    "guardado, protegido) do universo completo, para comparação.",
    casos_11_10,
    {"leituras_universo_completo": L2, "guardas_chave_fantasma": G},
    nota="**Retrata** a pasta exploratória `11.10_leituras_get_available_documents` anterior (17/09, fora do "
         "notebook, sem pré-registro): aquela usava lista de palavras inventada sobre a saída do "
         "`ConversationAgent` e reportava '2 confirmados' sem contar o universo completo, e mencionava o "
         "mecanismo de guarda sobre chave fantasma sem quantificá-lo. Esta célula substitui aquele conteúdo — "
         "mesma pasta, conteúdo reconciliado, os dois mecanismos quantificados exaustivamente.")

# não decide destino/impact (diferente da nº10/§11.9) — só documenta; refaz o registro final com a nota disponível
REGISTROS = [registro_final(u, fn, ATRIB, EU, SCH, AUTOCORR, PRESENCA, EST, STATUS, CONF_P6, STEPS_P1,
                            desfecho={**DESFECHO_11_9, **DESFECHO_11_10}) for u, fn in CANDIDATAS]
with open("resultados/unidades_memoria.json", "w", encoding="utf-8") as f:
    json.dump(REGISTROS, f, ensure_ascii=False, indent=2)
print("\nunidades_memoria.json regravado com o desfecho da §11.9 (nº10) e a nota da §11.10 (nº2):")
print(json.dumps(encurtar_ids(REGISTROS), ensure_ascii=False, indent=2))






erros já conhecidos (nº2, get_available_documents): 91 — usado só para conferência, não redefinido aqui.


<unknown>:40: SyntaxWarning: invalid escape sequence '\d'
<unknown>:24: SyntaxWarning: invalid escape sequence '\p'



total de leituras rastreadas (universo completo, todas as 635 execuções que declaram get_available_documents): 597


leituras
real  guardado protegido          
False False    False           110
      True     False            78
True  False    False           320
               True              1
      True     False            88


chave/índice errado, sem guarda de tipo e sem try/except — visível (já conta nos 91) vs. silencioso (novo):


...


>>> 4 ocorrências SILENCIOSAS confirmadas (leitura errada, sem guarda, sem try/except, sem exceção no step, fora dos 91 erros já conhecidos) — universo exaustivo, não amostra.


,exec_id,role,idx_chamada,idx_leitura,mes,forma,chave,tem_final_manager,degenerado_no_final_real,none_ou_null_no_final_real,parece_dict_no_final_real
47,175cd9f2…,ConversationAgent,0,3,2025-11,[],0,True,False,False,False
82,26e300f1…,ConversationAgent,0,2,2025-11,[],0,True,False,False,False
364,910fde1e…,ConversationAgent,0,0,2026-06,.get,documents,True,False,False,False
403,a47d6e3b…,ConversationAgent,0,3,2026-05,[],0,True,False,False,False



>>> Cruzando com DEGENERADO (único detector de resposta ruim já validado, sem lista de palavras nova): 0/4 têm a resposta REAL do managerAgent marcada como degenerada.
>>> Checagem adicional (None/null literal, ou dict/objeto aparentemente impresso) na MESMA resposta real: 0/4 com None/null · 0/4 parecendo dict impresso.
>>> Pela régua já validada do pipeline, e por nenhuma das duas checagens adicionais, nenhuma das ocorrências
>>> acima é hoje contável como 'resposta corrompida'. Isso RETRATA o rótulo 'confirmado: chega a final_answer
>>> errado' da pasta exploratória 11.10 anterior — aquela leitura usava lista de palavras inventada sobre a
>>> saída do ConversationAgent, não a resposta de verdade do managerAgent, nem um detector já validado.

guardas sobre chave fantasma achadas (qualquer desfecho): 1


,,ocorrências
chave_testada,descarta_documentos,
documents,True,1



>>> 1 ocorrência(s), no trace inteiro, onde o ramo que SEMPRE roda ignora a variável por completo (documentos reais descartados, sem virar valor errado).
  15f6ad52… chave='documents' mes=2026-05 | DEGENERADO no final real: False | None/null: False | parece dict: False
>>> Abaixo do piso de recorrência que qualquer candidata a memória precisa (≥3 execuções e ≥2 meses, §7 Passo 5) — 1 execução(ões), 1 mês(es). Não vira unidade própria; fica documentado, não pendurado.

>>> analise_funda (nº2): §11.10 (17/09): universo exaustivo (não amostra) — 4 ocorrências de leitura silenciosa tipo ".get sem plano B" (chave errada, sem guarda, sem try/except, sem exceção) em 4 execuções, 3 meses; 0/4 degeneradas por DEGENERADO, 0/4 com None/null literal, 0/4 parecendo dict impresso — três checagens independentes na resposta REAL do managerAgent, nenhuma achou corrupção; não confirma dano ao usuário final; retrata o 'confirmado' da pasta exploratória anterior. Segundo mecanismo, também quantificado: g